In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import sentencepiece as spm
import math
import os
from huggingface_hub import hf_hub_download

# ==========================================
# 1. CONFIGURATION (कॉन्फ़िगरेशन)
# ==========================================
class Config:
    D_MODEL              = 256
    N_HEADS              = 8
    N_ENC_LAYERS         = 4
    N_MEANING_DEC_LAYERS = 4
    N_DOHA_DEC_LAYERS    = 4
    D_FF                 = 1024
    DROPOUT              = 0.15
    MAX_SEQ_LEN          = 256
    MAX_MEANING_LEN      = 60
    MAX_DOHA_LEN         = 48
    PAD_ID               = 0
    BOS_ID               = 2
    EOS_ID               = 3
    # Generation Params
    GEN_TEMPERATURE      = 0.8
    GEN_TOP_K            = 50
    GEN_TOP_P            = 0.92
    GEN_REP_PENALTY      = 1.3
    GEN_DOHA_REP_PEN     = 1.5

cfg = Config()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HF_REPO_ID = "nikpatidar333/doha-generation-model_v2" # Stage 2 v3 Repo

# ==========================================
# 2. MODEL ARCHITECTURE (मॉडल संरचना)
# ==========================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

class DohaDecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model); self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model); self.norm4 = nn.LayerNorm(d_model)
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.cross_attn_enc = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.cross_attn_meaning = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(d_ff, d_model), nn.Dropout(dropout))
        self.gate = nn.Parameter(torch.tensor(0.5))
    def forward(self, x, encoder_memory, meaning_memory, tgt_mask, enc_key_padding_mask, meaning_key_padding_mask):
        res = x; x = self.norm1(x)
        attn_out, _ = self.self_attn(x, x, x, attn_mask=tgt_mask)
        x = res + attn_out
        res = x; x_norm = self.norm2(x)
        enc_out, _ = self.cross_attn_enc(x_norm, encoder_memory, encoder_memory, key_padding_mask=enc_key_padding_mask)
        meaning_out, _ = self.cross_attn_meaning(x_norm, meaning_memory, meaning_memory, key_padding_mask=meaning_key_padding_mask)
        g = torch.sigmoid(self.gate)
        x = res + g * enc_out + (1.0 - g) * meaning_out
        res = x; x = self.norm3(x)
        x = res + self.ffn(x)
        return self.norm4(x)

class UnifiedDohaModel(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_enc_layers, n_meaning_dec_layers, n_doha_dec_layers, d_ff, dropout, max_len, pad_id):
        super().__init__()
        self.pad_id = pad_id; self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_enc = PositionalEncoding(d_model, max_len, dropout)
        enc_l = nn.TransformerEncoderLayer(d_model, n_heads, d_ff, dropout, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_l, num_layers=n_enc_layers)
        m_dec_l = nn.TransformerDecoderLayer(d_model, n_heads, d_ff, dropout, batch_first=True, norm_first=True)
        self.meaning_decoder = nn.TransformerDecoder(m_dec_l, num_layers=n_meaning_dec_layers)
        self.doha_dec_layers = nn.ModuleList([DohaDecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_doha_dec_layers)])
        self.doha_dec_norm = nn.LayerNorm(d_model)
        self.meaning_proj = nn.Linear(d_model, vocab_size, bias=False)
        self.doha_proj = nn.Linear(d_model, vocab_size, bias=False)
        self.meaning_proj.weight = self.embedding.weight
        self.doha_proj.weight = self.embedding.weight

    def encode(self, src, src_mask):
        x = self.pos_enc(self.embedding(src) * math.sqrt(self.d_model))
        return self.encoder(x, src_key_padding_mask=~src_mask)

    def decode_meaning(self, tgt, memory, src_mask):
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1)).to(tgt.device)
        x = self.pos_enc(self.embedding(tgt) * math.sqrt(self.d_model))
        return self.meaning_decoder(x, memory, tgt_mask=tgt_mask, memory_key_padding_mask=~src_mask)

    def decode_doha(self, tgt, enc_mem, mean_mem, enc_mask, mean_mask):
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1)).to(tgt.device)
        x = self.pos_enc(self.embedding(tgt) * math.sqrt(self.d_model))
        for layer in self.doha_dec_layers:
            x = layer(x, enc_mem, mean_mem, tgt_mask, ~enc_mask, ~mean_mask)
        return self.doha_dec_norm(x)

# ==========================================
# 3. DOWNLOAD & LOAD (डाउनलोड और लोड)
# ==========================================
print("--- Downloading Stage 2 v3 Model ---")
MODEL_PATH = hf_hub_download(repo_id=HF_REPO_ID, filename="best_model.pt")
TOKENIZER_PATH = hf_hub_download(repo_id=HF_REPO_ID, filename="tokenizer.model")

sp = spm.SentencePieceProcessor()
sp.load(TOKENIZER_PATH)
VOCAB_SIZE = sp.get_piece_size()
DANDAA_ID  = sp.piece_to_id('॥')
STOP_TOKENS = {cfg.EOS_ID, DANDAA_ID}

model = UnifiedDohaModel(VOCAB_SIZE, cfg.D_MODEL, cfg.N_HEADS, cfg.N_ENC_LAYERS, 
                         cfg.N_MEANING_DEC_LAYERS, cfg.N_DOHA_DEC_LAYERS, cfg.D_FF, 
                         cfg.DROPOUT, cfg.MAX_SEQ_LEN, cfg.PAD_ID).to(DEVICE)

ckpt = torch.load(MODEL_PATH, map_location=DEVICE)
# DataParallel handle
state_dict = {k.replace('module.', ''): v for k, v in ckpt['model_state'].items()}
model.load_state_dict(state_dict)
model.eval()
print(f"--- Model Loaded (Vocab: {VOCAB_SIZE}) ---")

# ==========================================
# 5. RUN BATCH TESTS
# ==========================================
print("\n" + "="*60)
print(" APPROACH 1 — Forced Comma at 13-matra boundary")
print("="*60)
for i, test in enumerate(ntc):
    meaning, doha    = generate_forced_comma(model, sp, test['theme'], test['context'])
    penalty, details = charan_matra_score(doha)
    print(f"\nTest {i+1} | Theme: {test['theme']} | Context: {test['context']}")
    print(f"  अर्थ  : {meaning}")
    print(f"  दोहा  : {doha}")
    for ln in (1, 2):
        c1  = details.get(f'line{ln}_c1', '?')
        c2  = details.get(f'line{ln}_c2', '?')
        pen = details.get(f'line{ln}_penalty', '?')
        print(f"  पंक्ति {ln}: चरण1={c1}/13  चरण2={c2}/11  "
              f"दंड={pen} {'✅' if pen == 0 else '⚠️'}")
    print("-" * 40)

print("\n" + "="*60)
print(" APPROACH 2 — Best-of-5 with charan matra scoring")
print("="*60)
for i, test in enumerate(ntc):
    best = generate_doha_best_of_n(model, sp, test['theme'], test['context'],
                                   n=5, verbose=True)

print("\nBatch testing complete. ✅")
# ==========================================
# 5. RUN BATCH TESTS (बैच टेस्ट)
# ==========================================
ntc = [
    {'theme': 'शृंगार', 'context': 'मोरपंखी बाल'},
    {'theme': 'सौंदर्य', 'context': 'नाभि का भँवर'},
    {'theme': 'नायिका', 'context': 'सुंदर चंचल नायिका'},
    {'theme': 'रीति', 'context': 'ब्रजभाषा की सीख'},
    {'theme': 'मोह', 'context': 'मोहक भौंहें'},
    {'theme': 'प्रेम', 'context': 'सच्चा प्रेम'},
    {'theme': 'ज्ञान', 'context': 'आत्मज्ञान जरूरी'},
    {'theme': 'आध्यात्म', 'context': 'परमात्मा का स्वरूप'},
    {'theme': 'भक्ति', 'context': 'दिल में भगवान'},
    {'theme': 'साधना', 'context': 'दिल से साधना'},
    {'theme': 'नीति', 'context': 'अंधेरे के बाद'},
    {'theme': 'सामाजिक', 'context': 'खतरा हर जगह'},
    {'theme': 'जीवन', 'context': 'खुशहाल जीवन'},
    {'theme': 'भाग्य', 'context': 'जीवन के दुख'},
    {'theme': 'वीर', 'context': 'बुराई का अंत'},
    {'theme': 'ईश्वर', 'context': 'ईश्वर का साथ'},
    {'theme': 'कर्म', 'context': 'कर्म का फल'},
    {'theme': 'गुरु', 'context': 'गुरु से ज्ञान'},
    {'theme': 'धर्म', 'context': 'धर्म का काम'},
    {'theme': 'मिलन', 'context': 'प्यार का मिलन'},
    {'theme': 'त्याग', 'context': 'त्याग का महत्व'},
    {'theme': 'विरह', 'context': 'जुदाई का दुख'},
    {'theme': 'करुण', 'context': 'दुख में साथ'},
    {'theme': 'वैराग्य', 'context': 'जीवन नश्वर'},
    {'theme': 'शत्रुता', 'context': 'नफरत फैलाना'},
    {'theme': 'राजनीति', 'context': 'जनता का दुख'},
    {'theme': 'प्रकृति', 'context': 'वसंत की सुंदरता'},
    {'theme': 'ऋतु', 'context': 'फागुन की मस्ती'},
    {'theme': 'दर्शन', 'context': 'जीवन का चक्र'},
    {'theme': 'शिक्षा', 'context': 'कविता का ज्ञान'},
    {'theme': 'शांत', 'context': 'खुशियों भरा साल'},
    {'theme': 'संत', 'context': 'सच्चा संत निडर'},
    {'theme': 'दान', 'context': 'गुप्त दान महान'},
    {'theme': 'स्वभाव', 'context': 'दुर्जन नहीं बदलते'},
    {'theme': 'उपदेश', 'context': 'खुद की रक्षा करो'},
    {'theme': 'मोक्ष', 'context': 'मुक्ति चाहिए'},
    {'theme': 'संयम', 'context': 'इंद्रिय संयम'},
    {'theme': 'अध्यात्म', 'context': 'कपटी भक्ति बेकार'},
    {'theme': 'रूप', 'context': 'अद्भुत रूप दर्शन'},
    {'theme': 'आनंद', 'context': 'प्रभु मिलन आनंद'},
    {'theme': 'कृष्ण', 'context': 'कृष्ण को बुलाओ'},
    {'theme': 'धैर्य', 'context': 'नहीं रुकना'},
    {'theme': 'मित्रता', 'context': 'सच्चा दोस्त'},
    {'theme': 'अद्भुत', 'context': 'कविता ही दौलत'},
    {'theme': 'आशा', 'context': 'उम्मीद की किरण'},
    {'theme': 'संतोष', 'context': 'संतोषी जीवन'},
    {'theme': 'माया', 'context': 'आशा और तृष्णा'},
    {'theme': 'मृत्यु', 'context': 'मृत्यु अटल सत्य'},
    {'theme': 'समय', 'context': 'आज का काम अभी'},
    {'theme': 'विवेक', 'context': 'सही जगह बात करो'}
]

print("\n" + "="*60)
print(" RUNNING STAGE 2 v3 BATCH TESTS ")
print("="*60)

for i, test in enumerate(ntc):
    meaning, doha = generate(model, sp, test['theme'], test['context'])
    print(f"\nTest {i+1}:")
    print(f"Theme   : {test['theme']}")
    print(f"Context : {test['context']}")
    print(f"Meaning : {meaning}")
    print(f"Doha    : {doha}")
    print("-" * 40)

print("\nBatch testing complete. ✅")

--- Downloading Stage 2 v3 Model ---


best_model.pt:   0%|          | 0.00/152M [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/415k [00:00<?, ?B/s]

/tmp/ipykernel_55/3915307428.py:84: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_l, num_layers=n_enc_layers)


--- Model Loaded (Vocab: 8000) ---

 APPROACH 1 — Forced Comma at 13-matra boundary


NameError: name 'ntc' is not defined

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import sentencepiece as spm
import math
import os
from huggingface_hub import hf_hub_download

# ==========================================
# 1. CONFIGURATION (कॉन्फ़िगरेशन)
# ==========================================
class Config:
    D_MODEL              = 256
    N_HEADS              = 8
    N_ENC_LAYERS         = 4
    N_MEANING_DEC_LAYERS = 4
    N_DOHA_DEC_LAYERS    = 4
    D_FF                 = 1024
    DROPOUT              = 0.15
    MAX_SEQ_LEN          = 256
    MAX_MEANING_LEN      = 60
    MAX_DOHA_LEN         = 48
    PAD_ID               = 0
    BOS_ID               = 2
    EOS_ID               = 3
    # Generation Params
    GEN_TEMPERATURE      = 0.8
    GEN_TOP_K            = 50
    GEN_TOP_P            = 0.92
    GEN_REP_PENALTY      = 1.3
    GEN_DOHA_REP_PEN     = 1.5

cfg = Config()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HF_REPO_ID = "nikpatidar333/doha-generation-model_v2" # Stage 2 v3 Repo

# ==========================================
# 2. MODEL ARCHITECTURE (मॉडल संरचना)
# ==========================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

class DohaDecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model); self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model); self.norm4 = nn.LayerNorm(d_model)
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.cross_attn_enc = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.cross_attn_meaning = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(d_ff, d_model), nn.Dropout(dropout))
        self.gate = nn.Parameter(torch.tensor(0.5))
    def forward(self, x, encoder_memory, meaning_memory, tgt_mask, enc_key_padding_mask, meaning_key_padding_mask):
        res = x; x = self.norm1(x)
        attn_out, _ = self.self_attn(x, x, x, attn_mask=tgt_mask)
        x = res + attn_out
        res = x; x_norm = self.norm2(x)
        enc_out, _ = self.cross_attn_enc(x_norm, encoder_memory, encoder_memory, key_padding_mask=enc_key_padding_mask)
        meaning_out, _ = self.cross_attn_meaning(x_norm, meaning_memory, meaning_memory, key_padding_mask=meaning_key_padding_mask)
        g = torch.sigmoid(self.gate)
        x = res + g * enc_out + (1.0 - g) * meaning_out
        res = x; x = self.norm3(x)
        x = res + self.ffn(x)
        return self.norm4(x)

class UnifiedDohaModel(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_enc_layers, n_meaning_dec_layers, n_doha_dec_layers, d_ff, dropout, max_len, pad_id):
        super().__init__()
        self.pad_id = pad_id; self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_enc = PositionalEncoding(d_model, max_len, dropout)
        enc_l = nn.TransformerEncoderLayer(d_model, n_heads, d_ff, dropout, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_l, num_layers=n_enc_layers)
        m_dec_l = nn.TransformerDecoderLayer(d_model, n_heads, d_ff, dropout, batch_first=True, norm_first=True)
        self.meaning_decoder = nn.TransformerDecoder(m_dec_l, num_layers=n_meaning_dec_layers)
        self.doha_dec_layers = nn.ModuleList([DohaDecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_doha_dec_layers)])
        self.doha_dec_norm = nn.LayerNorm(d_model)
        self.meaning_proj = nn.Linear(d_model, vocab_size, bias=False)
        self.doha_proj = nn.Linear(d_model, vocab_size, bias=False)
        self.meaning_proj.weight = self.embedding.weight
        self.doha_proj.weight = self.embedding.weight

    def encode(self, src, src_mask):
        x = self.pos_enc(self.embedding(src) * math.sqrt(self.d_model))
        return self.encoder(x, src_key_padding_mask=~src_mask)

    def decode_meaning(self, tgt, memory, src_mask):
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1)).to(tgt.device)
        x = self.pos_enc(self.embedding(tgt) * math.sqrt(self.d_model))
        return self.meaning_decoder(x, memory, tgt_mask=tgt_mask, memory_key_padding_mask=~src_mask)

    def decode_doha(self, tgt, enc_mem, mean_mem, enc_mask, mean_mask):
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1)).to(tgt.device)
        x = self.pos_enc(self.embedding(tgt) * math.sqrt(self.d_model))
        for layer in self.doha_dec_layers:
            x = layer(x, enc_mem, mean_mem, tgt_mask, ~enc_mask, ~mean_mask)
        return self.doha_dec_norm(x)

# ==========================================
# 3. DOWNLOAD & LOAD (डाउनलोड और लोड)
# ==========================================
print("--- Downloading Stage 2 v3 Model ---")
MODEL_PATH = hf_hub_download(repo_id=HF_REPO_ID, filename="best_model.pt")
TOKENIZER_PATH = hf_hub_download(repo_id=HF_REPO_ID, filename="tokenizer.model")

sp = spm.SentencePieceProcessor()
sp.load(TOKENIZER_PATH)
VOCAB_SIZE = sp.get_piece_size()
DANDAA_ID  = sp.piece_to_id('॥')
STOP_TOKENS = {cfg.EOS_ID, DANDAA_ID}

model = UnifiedDohaModel(VOCAB_SIZE, cfg.D_MODEL, cfg.N_HEADS, cfg.N_ENC_LAYERS, 
                         cfg.N_MEANING_DEC_LAYERS, cfg.N_DOHA_DEC_LAYERS, cfg.D_FF, 
                         cfg.DROPOUT, cfg.MAX_SEQ_LEN, cfg.PAD_ID).to(DEVICE)

ckpt = torch.load(MODEL_PATH, map_location=DEVICE)
# DataParallel handle
state_dict = {k.replace('module.', ''): v for k, v in ckpt['model_state'].items()}
model.load_state_dict(state_dict)
model.eval()
print(f"--- Model Loaded (Vocab: {VOCAB_SIZE}) ---")

# ==========================================
# 4. SAMPLING & GENERATION LOGIC
# ==========================================
def top_k_top_p_sample(logits, temperature=0.8, top_k=50, top_p=0.92, past_ids=None, rep_penalty=1.3):
    logits = logits.squeeze(0).float() / temperature
    if past_ids and rep_penalty > 1.0:
        for tid in set(past_ids[-20:]):
            logits[tid] /= rep_penalty if logits[tid] > 0 else (1/rep_penalty)
    if top_k > 0:
        v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
        logits[logits < v[-1]] = float('-inf')
    if top_p < 1.0:
        p_sort, p_idx = torch.sort(F.softmax(logits, dim=-1), descending=True)
        cumsum = torch.cumsum(p_sort, dim=-1)
        p_sort[(cumsum - p_sort) > top_p] = 0.0
        next_token = torch.multinomial(p_sort, 1)
        return p_idx.gather(-1, next_token)
    return torch.multinomial(F.softmax(logits, dim=-1), 1)

@torch.no_grad()
def generate(model, sp, theme, context):
    # Phase 1: Meaning
    enc_text = f"<theme> {theme} </theme> <context> {context} </context>"
    enc_ids = torch.tensor([sp.encode(enc_text)], dtype=torch.long).to(DEVICE)
    enc_mask = (enc_ids != cfg.PAD_ID)
    enc_mem = model.encode(enc_ids, enc_mask)
    
    mean_ids = [cfg.BOS_ID]
    for _ in range(cfg.MAX_MEANING_LEN):
        m_in = torch.tensor([mean_ids], dtype=torch.long).to(DEVICE)
        m_out = model.decode_meaning(m_in, enc_mem, enc_mask)
        next_id = top_k_top_p_sample(model.meaning_proj(m_out[:, -1, :]), 
                                     past_ids=mean_ids, rep_penalty=cfg.GEN_REP_PENALTY).item()
        if next_id == cfg.EOS_ID: break
        mean_ids.append(next_id)
    gen_meaning = sp.decode(mean_ids[1:])
    
    # Phase 2: Doha
    enc_text_full = f"{enc_text} <meaning> {gen_meaning} </meaning>"
    enc_ids_full = torch.tensor([sp.encode(enc_text_full)], dtype=torch.long).to(DEVICE)
    enc_mask_full = (enc_ids_full != cfg.PAD_ID)
    enc_mem_full = model.encode(enc_ids_full, enc_mask_full)
    
    m_mem = model.decode_meaning(torch.tensor([mean_ids], dtype=torch.long).to(DEVICE), 
                                  enc_mem_full, enc_mask_full)
    m_mask = (torch.tensor([mean_ids]) != cfg.PAD_ID).to(DEVICE)
    
    doha_ids = [cfg.BOS_ID]
    for _ in range(cfg.MAX_DOHA_LEN):
        d_in = torch.tensor([doha_ids], dtype=torch.long).to(DEVICE)
        d_out = model.decode_doha(d_in, enc_mem_full, m_mem, enc_mask_full, m_mask)
        next_id = top_k_top_p_sample(model.doha_proj(d_out[:, -1, :]), 
                                     past_ids=doha_ids, rep_penalty=cfg.GEN_DOHA_REP_PEN).item()
        doha_ids.append(next_id)
        if next_id in STOP_TOKENS: break
        
    return gen_meaning, sp.decode(doha_ids[1:])
# ── Matra counting ──────────────────────────────────────────
def count_matras(text):
    """Count matras in a Hindi text string."""
    # Long vowels — 2 matras
    long_vowels = set('आईऊएऐओऔाीूेैोौ')
    # Short vowels — 1 matra
    short_vowels = set('अइउऋिुृ')
    # Anusvara, visarga — 1 matra
    other_matras = set('ंःँ')

    count = 0
    for ch in text:
        if ch in long_vowels:
            count += 2
        elif ch in short_vowels or ch in other_matras:
            count += 1
        elif '\u0900' <= ch <= '\u097F':
            # Any other Devanagari char (consonants etc) = 1 matra
            count += 1
    return count


def matra_score(doha_text):
    """
    Returns (penalty, line1_matras, line2_matras).
    penalty=0 means perfect 24 matras per line.
    Lower penalty = better.
    """
    # Clean and split into lines
    text = doha_text.replace('॥', '').strip()
    # Try newline split first, then comma
    if '\n' in text:
        lines = [l.strip() for l in text.split('\n') if l.strip()]
    elif ',' in text:
        lines = [l.strip() for l in text.split(',') if l.strip()]
    else:
        lines = [text]

    if len(lines) < 2:
        return 999, 0, 0

    l1 = count_matras(lines[0])
    l2 = count_matras(lines[1])
    penalty = abs(l1 - 24) + abs(l2 - 24)
    return penalty, l1, l2


def generate_doha_best_of_n(model, sp, theme, context, n=5,
                              temperature=None, top_k=None, top_p=None):
    """
    Generate n dohas and return the one with best matra count.
    Target: 24 matras per line.
    """
    candidates = []
    for i in range(n):
        meaning, doha = generate_doha(
            model, sp, theme, context,
            temperature=temperature, top_k=top_k, top_p=top_p,
            verbose=False
        )
        penalty, l1, l2 = matra_score(doha)
        candidates.append({
            'doha'     : doha,
            'meaning'  : meaning,
            'penalty'  : penalty,
            'l1_matras': l1,
            'l2_matras': l2,
        })

    # Sort by matra penalty — lowest is best
    candidates.sort(key=lambda x: x['penalty'])
    best = candidates[0]

    print(f"थीम    : {theme}")
    print(f"संदर्भ : {context}")
    print(f"अर्थ   : {best['meaning']}")
    print(f"दोहा   : {best['doha']}")
    print(f"मात्रा : पंक्ति1={best['l1_matras']}/24  पंक्ति2={best['l2_matras']}/24  "
          f"दंड={best['penalty']}")
    print(f"सभी {n} प्रयास:")
    for j, c in enumerate(candidates):
        marker = '★' if j == 0 else ' '
        print(f"  {marker} [{j+1}] penalty={c['penalty']} "
              f"({c['l1_matras']}/{c['l2_matras']}) : {c['doha'][:60]}")
    print("-" * 60)

    return best


print("Matra scoring + best-of-N generation ready ✅")
# ==========================================
# 5. RUN BATCH TESTS (बैच टेस्ट)
# ==========================================
ntc_extended = [
    # --- जीवन और दर्शन (Life & Philosophy) ---
    {'theme': 'संघर्ष', 'context': 'पत्थर का चोट सहकर मूरत बनना'},
    {'theme': 'त्याग', 'context': 'दीपक का खुद जलकर दूसरों को प्रकाश देना'},
    {'theme': 'बदलाव', 'context': 'पुरानी केंचुली छोड़ता हुआ सर्प'},
    {'theme': 'नश्वरता', 'context': 'पानी के बुलबुले का क्षण भर में मिट जाना'},
    {'theme': 'शांति', 'context': 'भीड़भाड़ के बीच मन का मौन हो जाना'},
    {'theme': 'अनुभव', 'context': 'सफेद बालों में छिपे जीवन के गहरे सबक'},
    {'theme': 'भाग्य', 'context': 'समुद्र किनारे लहरों का इंतज़ार करता नाविक'},
    {'theme': 'अस्तित्व', 'context': 'रेत पर बने पैरों के निशान और आती लहरें'},
    {'theme': 'शून्यता', 'context': 'रात के सन्नाटे में ब्रह्मांड की गूँज'},
    {'theme': 'संतुलन', 'context': 'साइकिल के पहिये का निरंतर घूमते रहना'},

    # --- प्रकृति और प्रतीक (Nature & Symbols) ---
    {'theme': 'आशा', 'context': 'रेगिस्तान के बीचों-बीच खिलता हुआ एक फूल'},
    {'theme': 'उदारता', 'context': 'बादलों का बिना भेदभाव के बंजर पर बरसना'},
    {'theme': 'तटस्थता', 'context': 'कमल के पत्ते पर पानी की बूंद का न ठहरना'},
    {'theme': 'दृढ़ता', 'context': 'तूफान में भी अडिग खड़ा हिमालय'},
    {'theme': 'मर्यादा', 'context': 'समुद्र का अपनी सीमाओं को कभी न लांघना'},
    {'theme': 'स्वतंत्रता', 'context': 'पिंजरा तोड़कर खुले आकाश में उड़ता पक्षी'},
    {'theme': 'धैर्य', 'context': 'पतझड़ के बाद नई कोंपलों का धैर्यपूर्ण इंतज़ार'},
    {'theme': 'पुनर्जन्म', 'context': 'राख से उठती हुई फीनिक्स की उड़ान'},
    {'theme': 'शक्ति', 'context': 'चट्टानों का सीना चीरकर बहता झरना'},
    {'theme': 'अनंत', 'context': 'क्षितिज जहाँ धरती और आकाश मिलते प्रतीत होते हैं'},

    # --- मानवीय भावनाएँ (Human Emotions) ---
    {'theme': 'ममता', 'context': 'चिड़िया का अपने बच्चों के लिए तिनका-तिनका जोड़ना'},
    {'theme': 'क्रोध', 'context': 'सुलगता हुआ कोयला जो खुद को पहले जलाता है'},
    {'theme': 'ईर्ष्या', 'context': 'दीमक की तरह अंदर ही अंदर खोखला करना'},
    {'theme': 'करुणा', 'context': 'घायल पक्षी को गोद में उठाकर मरहम लगाना'},
    {'theme': 'उत्साह', 'context': 'भोर की पहली किरण के साथ नई शुरुआत'},
    {'theme': 'पछतावा', 'context': 'चिड़िया के खेत चुग जाने के बाद रखवाली करना'},
    {'theme': 'भय', 'context': 'अंधेरी रात में अपनी ही परछाईं से डरना'},
    {'theme': 'प्रेम', 'context': 'चाँदनी का रात से बिना शर्त जुड़ाव'},
    {'theme': 'एकाकीपन', 'context': 'भरी महफिल में खुद को अकेला पाना'},
    {'theme': 'विस्मय', 'context': 'छोटे शिशु की आँखों में दुनिया का कौतूहल'},

    # --- नैतिक मूल्य (Ethics & Values) ---
    {'theme': 'विनम्रता', 'context': 'फलों से लदी टहनियों का जमीन की ओर झुक जाना'},
    {'theme': 'क्षमा', 'context': 'चंदन के वृक्ष का कुल्हाड़ी को भी सुगंधित करना'},
    {'theme': 'कृतज्ञता', 'context': 'धूप में छाया देने वाले वृक्ष को न काटना'},
    {'theme': 'ईमानदारी', 'context': 'अंधेरे में भी सही रास्ते पर चलने का साहस'},
    {'theme': 'कर्तव्य', 'context': 'सीमा पर डटे सैनिक की अडिग निष्ठा'},
    {'theme': 'संयम', 'context': 'उफनती नदी का तटों की मर्यादा में रहना'},
    {'theme': 'एकता', 'context': 'तिनकों से बनी रस्सी का हाथी को बांध लेना'},
    {'theme': 'सत्य', 'context': 'बादलों की ओट में छिपा सूर्य जो देर-सबेर निकलेगा'},
    {'theme': 'परोपकार', 'context': 'मधुमक्खी का दूसरों के लिए शहद इकठ्ठा करना'},
    {'theme': 'श्रद्धा', 'context': 'पत्थर की मूरत में भी ईश्वर का वास देखना'},

    # --- चुनौतियाँ और सफलता (Challenges & Success) ---
    {'theme': 'परिश्रम', 'context': 'चींटी का अपने वजन से भारी दाना पहाड़ पर ले जाना'},
    {'theme': 'एकाग्रता', 'context': 'अर्जुन की दृष्टि और केवल मछली की आँख'},
    {'theme': 'लक्ष्य', 'context': 'बाण का धनुष से छूटकर निशाने की ओर बढ़ना'},
    {'theme': 'साधना', 'context': 'बूँद-बूँद गिरने से कठोर पत्थर पर पड़ा निशान'},
    {'theme': 'जीत', 'context': 'हजार बार गिरने के बाद एक बार फिर खड़े होना'},
    {'theme': 'अवसर', 'context': 'दस्तक देती हवा जिसे पहचानना जरूरी है'},
    {'theme': 'रणनीति', 'context': 'बिसात पर चाल चलने से पहले का मौन चिंतन'},
    {'theme': 'दृष्टिकोण', 'context': 'आधे भरे गिलास को पूरा देखना'},
    {'theme': 'प्रतिभा', 'context': 'कीचड़ के बीच भी अपनी आभा बिखेरता कमल'},
    {'theme': 'अनुशासन', 'context': 'सेना की तालबद्ध कदमताल'},

    # --- रिश्ते और समाज (Relationships & Society) ---
    {'theme': 'दोस्ती', 'context': 'कृष्ण और सुदामा के कच्चे चावलों की पोटली'},
    {'theme': 'विश्वास', 'context': 'अंधेरे में भी पिता का हाथ थामे चलता बच्चा'},
    {'theme': 'संस्कार', 'context': 'जड़ों की मजबूती जो पेड़ को गिरने नहीं देती'},
    {'theme': 'लोभ', 'context': 'सोने के अंडे देने वाली मुर्गी को मार देना'},
    {'theme': 'स्वार्थ', 'context': 'काम निकलने पर साये का भी साथ छोड़ देना'},
    {'theme': 'सम्मान', 'context': 'बुजुर्गों की झुर्रियों में छिपे ज्ञान का आदर'},
    {'theme': 'विवाद', 'context': 'एक ही बात के दो अलग-अलग छोरों पर खड़े होना'},
    {'theme': 'परंपरा', 'context': 'मिट्टी के दीये की लौ जो पीढ़ी-दर-पीढ़ी जलती है'},
    {'theme': 'घृणा', 'context': 'हाथ में कीचड़ लेकर दूसरों पर फेंकने की कोशिश'},
    {'theme': 'अतिथि', 'context': 'द्वार पर आए अंजान में देवत्व की खोज'},

    # --- ज्ञान और बुद्धि (Knowledge & Wisdom) ---
    {'theme': 'जिज्ञासा', 'context': 'पत्ते के गिरने के पीछे का "क्यों" ढूँढना'},
    {'theme': 'अज्ञान', 'context': 'दीये के नीचे का घना अंधेरा'},
    {'theme': 'विवेक', 'context': 'हंस का दूध और पानी को अलग कर देना'},
    {'theme': 'सादगी', 'context': 'मिट्टी के घड़े के शीतल जल का आनंद'},
    {'theme': 'स्मृति', 'context': 'पुरानी धूल भरी तस्वीरों में कैद मुस्कान'},
    {'theme': 'भ्रम', 'context': 'रेगिस्तान में मृगजल की अंतहीन तलाश'},
    {'theme': 'मौन', 'context': 'शब्दों के समाप्त होने पर सत्य का उदय'},
    {'theme': 'साक्षरता', 'context': 'काले अक्षरों में छिपे संसार की चाबी पाना'},
    {'theme': 'तर्क', 'context': 'अंधेरे में मशाल जलाकर गड्ढे को देखना'},
    {'theme': 'प्रेरणा', 'context': 'बुझते दीये को दूसरी लौ से प्रज्वलित करना'},

    # --- समय और गति (Time & Motion) ---
    {'theme': 'प्रतीक्षा', 'context': 'शबरी के मीठे बेर और राम का इंतज़ार'},
    {'theme': 'निरंतरता', 'context': 'घड़ी की टिक-टिक जो कभी नहीं थकती'},
    {'theme': 'विश्राम', 'context': 'दिन भर की थकान के बाद माँ की गोद'},
    {'theme': 'बचपन', 'context': 'कागज की नाव और बारिश का पानी'},
    {'theme': 'बुढ़ापा', 'context': 'ढलते सूरज की शांत और सौम्य लालिमा'},
    {'theme': 'युवावस्था', 'context': 'उफनती लहरों की असीम ऊर्जा'},
    {'theme': 'कल', 'context': 'एक ऐसा सपना जो कभी हाथ नहीं आता'},
    {'theme': 'आज', 'context': 'मुट्ठी में बंद रेत जो धीरे-धीरे फिसल रही है'},
    {'theme': 'क्षण', 'context': 'कैमरे की एक क्लिक में कैद पूरी जिंदगी'},
    {'theme': 'इतिहास', 'context': 'खंडहरों की दीवारों पर लिखी वीरगाथाएं'},

    # --- अन्य महत्वपूर्ण विषय (Miscellaneous) ---
    {'theme': 'खोज', 'context': 'अंधेरी गुफा में मशाल लेकर हीरे की तलाश'},
    {'theme': 'ममता', 'context': 'गाय का अपने बछड़े को चाटना'},
    {'theme': 'लचीलापन', 'context': 'हवा के झोंके में झुकती घास जो टूटती नहीं'},
    {'theme': 'मजबूरी', 'context': 'पिंजरे में बंद शेर का बेबस दहाड़ना'},
    {'theme': 'उत्सव', 'context': 'अमावस की रात में दीपों की कतार'},
    {'theme': 'अन्याय', 'context': 'ताकतवर भेड़िये का मेमने पर प्रहार'},
    {'theme': 'न्याय', 'context': 'तराजू के दोनों पलड़ों का बराबर होना'},
    {'theme': 'कला', 'context': 'कोरे कागज पर रंगों का जीवंत संवाद'},
    {'theme': 'साहस', 'context': 'अंधेरी सुरंग के अंत में प्रकाश की उम्मीद'},
    {'theme': 'निर्मलता', 'context': 'पहाड़ों से निकलता कांच जैसा साफ पानी'},
    {'theme': 'वफादारी', 'context': 'मालिक की चौखट पर बैठा बूढ़ा कुत्ता'},
    {'theme': 'गर्व', 'context': 'तिरंगे का आकाश की ऊंचाइयों में लहराना'},
    {'theme': 'सादगी', 'context': 'बिना गहनों के भी चेहरे का प्राकृतिक तेज'},
    {'theme': 'उलझन', 'context': 'धागे का ऐसा सिरा जो कहीं खो गया है'},
    {'theme': 'आकर्षण', 'context': 'पतंगे का जलती शमशाम की ओर खिंचे जाना'},
    {'theme': 'वैराग्य', 'context': 'राजपाट छोड़कर सिद्धार्थ का वन को जाना'},
    {'theme': 'अभिमान', 'context': 'रावण के दस सिरों का अपने ज्ञान पर गुमान'},
    {'theme': 'सुख', 'context': 'गर्मियों की दोपहर में घने पेड़ की छाँव'},
    {'theme': 'दुःख', 'context': 'सूखी आँखों से बहते अदृश्य आँसू'},
    {'theme': 'ईश्वर', 'context': 'हवा की तरह जो दिखती नहीं पर महसूस होती है'},

    # (इसी क्रम में अगले 100 विषय...)
    {'theme': 'मौन', 'context': 'शब्दों के थकने के बाद शुरू होने वाली बातचीत'},
    {'theme': 'क्रूरता', 'context': 'सूखे पेड़ को जड़ से उखाड़ देना'},
    {'theme': 'सुगंध', 'context': 'मिट्टी पर पहली बारिश की बूंदें'},
    {'theme': 'साक्षी', 'context': 'पुराना बरगद जिसने सदियां गुजरते देखीं'},
    {'theme': 'संगीत', 'context': 'बाँसुरी के छेदों से निकलता रूहानी स्वर'},
    {'theme': 'साहचर्य', 'context': 'धूप और छाँव का लुका-छिपी का खेल'},
    {'theme': 'कल्पना', 'context': 'बादलों में आकृतियां ढूँढता एक कवि'},
    {'theme': 'मर्यादा', 'context': 'लक्ष्मण रेखा जो सुरक्षा का घेरा है'},
    {'theme': 'तपस्या', 'context': 'बर्फानी चोटियों पर बिना वस्त्रों के साधु'},
    {'theme': 'विनाश', 'context': 'ज्वालामुखी का दहकता हुआ लावा'},
    {'theme': 'निर्माण', 'context': 'खंडहरों के बीच से नए शहर का उदय'},
    {'theme': 'प्रसिद्धि', 'context': 'आकाश में टूटते तारे की एक क्षण की चमक'},
    {'theme': 'गुमनामी', 'context': 'जंगल में खिले फूल की अनसुनी खुशबू'},
    {'theme': 'अपेक्षा', 'context': 'सूखे खेत का आसमान की ओर ताकना'},
    {'theme': 'संतुष्टि', 'context': 'भरपेट भोजन के बाद एक लंबी डकार'},
    {'theme': 'स्वप्न', 'context': 'बंद आँखों से सात समंदर पार की सैर'},
    {'theme': 'हकीकत', 'context': 'धूप में जलते नंगे पैरों के छाले'},
    {'theme': 'अधिकार', 'context': 'सिंहासन पर बैठते ही बदलती मानसिकता'},
    {'theme': 'अपराध', 'context': 'रात के अंधेरे में चेहरे पर लगा नकाब'},
    {'theme': 'भक्ति', 'context': 'हनुमान का सीना चीरकर राम का नाम दिखाना'},
    {'theme': 'बलिदान', 'context': 'देश के लिए हँसते-हँसते फांसी चढ़ना'},
    {'theme': 'चपलता', 'context': 'गिलहरी का पेड़ पर चढ़ना और उतरना'},
    {'theme': 'गंभीरता', 'context': 'गहरे पानी का शांत व्यवहार'},
    {'theme': 'संशय', 'context': 'दो रास्तों के बीच खड़ा दुविधा में फँसा राही'},
    {'theme': 'निर्भयता', 'context': 'शेर की मांद में घुसकर उसे चुनौती देना'},
    {'theme': 'आलस्य', 'context': 'दीवार पर टंगी रुकी हुई घड़ी'},
    {'theme': 'परिवर्तन', 'context': 'मौसम के साथ बदलता जंगलों का रंग'},
    {'theme': 'एकनिष्ठता', 'context': 'ध्रुव तारे का अपनी जगह पर अडिग रहना'},
    {'theme': 'लालच', 'context': 'मछली का कांटे में फँसे मांस के टुकड़े को निगलना'},
    {'theme': 'क्रान्ति', 'context': 'दबे हुए अंगारों का अचानक भभक उठना'},
    {'theme': 'सहनशीलता', 'context': 'धरती जो सबका बोझ सहती है'},
    {'theme': 'मिठास', 'context': 'गन्ने के रस का अंतिम बूंद तक मीठा होना'},
    {'theme': 'कठोरता', 'context': 'नारियल का बाहरी सख्त कवच'},
    {'theme': 'कोमलता', 'context': 'गुलाब की पंखुड़ियों पर ओस की बूंद'},
    {'theme': 'विछोह', 'context': 'रेलवे स्टेशन पर छूटता हुआ हाथ'},
    {'theme': 'मिलन', 'context': 'नदी का सागर की बाहों में समा जाना'},
    {'theme': 'चोरी', 'context': 'चाँद का सूरज की रोशनी चुराना'},
    {'theme': 'दान', 'context': 'कर्ण का अपने कवच और कुंडल उतार देना'},
    {'theme': 'धोखा', 'context': 'दूध में पानी का सफेद झूठ'},
    {'theme': 'अमृत', 'context': 'ज्ञान के मंथन से निकला निष्कर्ष'},
    {'theme': 'विष', 'context': 'कानों में घोली गई कड़वी बातें'},
    {'theme': 'वैभव', 'context': 'इंद्रधनुष के सात रंगों की सजावट'},
    {'theme': 'दरिद्रता', 'context': 'खाली थाली में चाँद का अक्स देखना'},
    {'theme': 'श्रृंगार', 'context': 'धरती पर फैली मखमली हरी घास'},
    {'theme': 'वैराग्य', 'context': 'श्मशान की राख और जीवन का सत्य'},
    {'theme': 'चेतना', 'context': 'पत्थर के भीतर छिपी हुई आग'},
    {'theme': 'विद्रोही', 'context': 'पिंजरे की तीलियां तोड़ता परिंदा'},
    {'theme': 'शरण', 'context': 'थके हुए राही के लिए सराय का दीया'},
    {'theme': 'उदारता', 'context': 'फलों का स्वयं न खाकर दूसरों को देना'},
    {'theme': 'अहंकार', 'context': 'रावण के सोने की लंका का दहन'},
    {'theme': 'ममता', 'context': 'यशौदा का कान्हा को माखन खिलाना'},
    {'theme': 'विरह', 'context': 'यक्ष की मेघदूत के माध्यम से पुकार'},
    {'theme': 'संदेह', 'context': 'दूध का जला मट्ठा भी फूंककर पीना'},
    {'theme': 'शांति', 'context': 'बुद्ध के चेहरे की सौम्य मुस्कान'},
    {'theme': 'संसार', 'context': 'एक बड़ा मेला जहाँ सब अजनबी हैं'},
    {'theme': 'मर्यादा', 'context': 'नदी का अपने किनारे न तोड़ना'},
    {'theme': 'विजेता', 'context': 'हार मानकर भी फिर से उठने वाला'},
    {'theme': 'पराजय', 'context': 'मैदान छोड़कर भागने का डर'},
    {'theme': 'साहस', 'context': 'अकेले ही भीड़ के खिलाफ खड़े होना'},
    {'theme': 'प्रकृति', 'context': 'पहाड़ों की चोटियों पर ढलती शाम'},
    {'theme': 'मानवता', 'context': 'दुश्मन को भी प्यास लगने पर पानी पिलाना'},
    {'theme': 'सहनशक्ति', 'context': 'लोहे का गर्म होकर हथौड़े की मार सहना'},
    {'theme': 'परिवेश', 'context': 'गाँव की चौपाल और नीम का पेड़'},
    {'theme': 'स्मृति', 'context': 'हवा में तैरती बचपन की खुशबू'},
    {'theme': 'उम्मीद', 'context': 'अंधेरे घर में जलता हुआ नन्हा दीया'},
    {'theme': 'विश्वासघात', 'context': 'आस्तीन में पाला हुआ सांप'},
    {'theme': 'करुणा', 'context': 'रास्ते के पत्थर को हटाना ताकि कोई गिरे नहीं'},
    {'theme': 'सौन्दर्य', 'context': 'पूर्णमासी का पूरा खिला हुआ चाँद'},
    {'theme': 'कष्ट', 'context': 'नंगे पैर कँटीली राहों पर चलना'},
    {'theme': 'सुविधा', 'context': 'सोने के पिंजरे में बंद पक्षी'},
    {'theme': 'अतृप्ति', 'context': 'रेगिस्तान में भटकता हुआ प्यासा'},
    {'theme': 'तृप्ति', 'context': 'मनचाहा वरदान मिलने की खुशी'},
    {'theme': 'अंतर्मन', 'context': 'दर्पण जो चेहरा नहीं चरित्र दिखाता है'},
    {'theme': 'बाहरी दिखावा', 'context': 'ऊपर से फिट और अंदर से अनफिट'},
    {'theme': 'साधुता', 'context': 'बिना किसी मोह के दुनिया से गुजर जाना'},
    {'theme': 'कुटिलता', 'context': 'शकुनि के पासे और बिछाई गई बिसात'},
    {'theme': 'चमक', 'context': 'अंधेरे में जुगनू की छोटी सी रोशनी'},
    {'theme': 'गहराई', 'context': 'समुद्र के तल में छिपे हुए मोती'},
    {'theme': 'ऊंचाई', 'context': 'ईगल की नजर जो बादलों के ऊपर है'},
    {'theme': 'विस्तार', 'context': 'फैला हुआ नीला अंबर जिसका अंत नहीं'},
    {'theme': 'संकीर्णता', 'context': 'कुएँ का मेंढक जो उसे ही दुनिया मानता है'},
    {'theme': 'स्थिरता', 'context': 'झील का पानी जिसमें लहरें नहीं हैं'},
    {'theme': 'चंचलता', 'context': 'पवन का झोंका जो यहाँ-वहाँ भटकता है'},
    {'theme': 'सृष्टि', 'context': 'मिट्टी से बने खिलौने और उनका रचयिता'},
    {'theme': 'प्रलय', 'context': 'शिव का तांडव और टूटता ब्रह्मांड'},
    {'theme': 'आस्था', 'context': 'कांच की दीवार के पार भी रास्ता देखना'},
    {'theme': 'तर्कशक्ति', 'context': 'बीरबल की खिचड़ी और उसकी बुद्धिमत्ता'},
    {'theme': 'लालसा', 'context': 'इंद्रधनुष को मुट्ठी में पकड़ने की कोशिश'},
    {'theme': 'निर्लेपता', 'context': 'संसार में रहकर भी संसार का न होना'},
    {'theme': 'विद्रूपता', 'context': 'सुंदर चेहरे के पीछे छिपा बदसूरत विचार'},
    {'theme': 'समर्पण', 'context': 'बाँसुरी का स्वयं को खाली कर देना'},
    {'theme': 'आकर्षण', 'context': 'लोहे का चुंबक की ओर खिंचना'},
    {'theme': 'प्रतिशोध', 'context': 'जंगल की आग जो सब भस्म कर दे'},
    {'theme': 'सत्यनिष्ठा', 'context': 'हरिश्चंद्र का श्मशान में कर मांगना'},
    {'theme': 'धैर्य', 'context': 'बर्फ का पिघलकर धीरे-धीरे पानी बनना'},
    {'theme': 'उत्सुकता', 'context': 'बंद लिफाफे में क्या है यह जानने की बेचैनी'},
    {'theme': 'अहंकार', 'context': 'पहाड़ का अपनी ऊंचाई पर इतराना'},
    {'theme': 'विनम्रता', 'context': 'नदी का सागर में मिलकर अपना अस्तित्व खोना'},
    {'theme': 'जीवन', 'context': 'सांसों की एक ऐसी माला जो कभी टूटती है'},
    {'theme': 'मृत्यु', 'context': 'एक लंबी नींद जिसके बाद कोई नहीं जागता'}
]

print("\n" + "="*60)
print(" RUNNING STAGE 2 v3 BATCH TESTS ")
print("="*60)

for i, test in enumerate(ntc):
    meaning, doha = generate(model, sp, test['theme'], test['context'])
    print(f"\nTest {i+1}:")
    print(f"Theme   : {test['theme']}")
    print(f"Context : {test['context']}")
    print(f"Meaning : {meaning}")
    print(f"Doha    : {doha}")
    print("-" * 40)

print("\nBatch testing complete. ✅")

--- Downloading Stage 2 v3 Model ---


/tmp/ipykernel_55/602308397.py:84: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_l, num_layers=n_enc_layers)


--- Model Loaded (Vocab: 8000) ---
Matra scoring + best-of-N generation ready ✅

 RUNNING STAGE 2 v3 BATCH TESTS 

Test 1:
Theme   : शृंगार
Context : मोरपंखी बाल
Meaning : नायिका के मन में मोर पँखुरी बाल (कृष्ण) की ओर हैं, जिससे मुख की चमक चमकते हुए हैं। पर मोतियों के सौंदर्य और आकर्षक रूपी श्याम (कृष्ण) की सुंदरता का वर्णन है।
Doha    : मोर उड़त मन खोलिन में ललित बालन ओर। मनो बरुन बाल के, छवि लाल स्याम ॥
----------------------------------------

Test 2:
Theme   : सौंदर्य
Context : नाभि का भँवर
Meaning : नायिका के शरीर में ही जल-लता जल जल-वास के समान है। बिना बिना किसी व्यक्ति से भी किसी का कोई नहीं, उसे कोई दोष नहीं छोड़ता, क्योंकि नाभि का अत्यंत सौंदर्य है।
Doha    : जल-जल तन में नीर है, बिना शरीर की आस। बिन जाने को कोई नहीं, कौन न जाहु अकाम ॥
----------------------------------------

Test 3:
Theme   : नायिका
Context : सुंदर चंचल नायिका
Meaning : नायिका के शरीर पर लालिमा का वर्णन करता है कि जो उसके नयनों में चमक रहे हैं और उसकी आँखों से नैन नयन से देखते हैं, वह अपनी छवि के समान सुंदर

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import sentencepiece as spm
import math
import os
from huggingface_hub import hf_hub_download

# ==========================================
# 1. CONFIGURATION (कॉन्फ़िगरेशन)
# ==========================================
class Config:
    D_MODEL              = 256
    N_HEADS              = 8
    N_ENC_LAYERS         = 4
    N_MEANING_DEC_LAYERS = 4
    N_DOHA_DEC_LAYERS    = 4
    D_FF                 = 1024
    DROPOUT              = 0.15
    MAX_SEQ_LEN          = 256
    MAX_MEANING_LEN      = 60
    MAX_DOHA_LEN         = 48
    PAD_ID               = 0
    BOS_ID               = 2
    EOS_ID               = 3
    # Generation Params
    GEN_TEMPERATURE      = 0.8
    GEN_TOP_K            = 50
    GEN_TOP_P            = 0.92
    GEN_REP_PENALTY      = 1.3
    GEN_DOHA_REP_PEN     = 1.5

cfg = Config()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HF_REPO_ID = "nikpatidar333/doha-generation-model_v2" # Stage 2 v3 Repo

# ==========================================
# 2. MODEL ARCHITECTURE (मॉडल संरचना)
# ==========================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

class DohaDecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model); self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model); self.norm4 = nn.LayerNorm(d_model)
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.cross_attn_enc = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.cross_attn_meaning = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(d_ff, d_model), nn.Dropout(dropout))
        self.gate = nn.Parameter(torch.tensor(0.5))
    def forward(self, x, encoder_memory, meaning_memory, tgt_mask, enc_key_padding_mask, meaning_key_padding_mask):
        res = x; x = self.norm1(x)
        attn_out, _ = self.self_attn(x, x, x, attn_mask=tgt_mask)
        x = res + attn_out
        res = x; x_norm = self.norm2(x)
        enc_out, _ = self.cross_attn_enc(x_norm, encoder_memory, encoder_memory, key_padding_mask=enc_key_padding_mask)
        meaning_out, _ = self.cross_attn_meaning(x_norm, meaning_memory, meaning_memory, key_padding_mask=meaning_key_padding_mask)
        g = torch.sigmoid(self.gate)
        x = res + g * enc_out + (1.0 - g) * meaning_out
        res = x; x = self.norm3(x)
        x = res + self.ffn(x)
        return self.norm4(x)

class UnifiedDohaModel(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_enc_layers, n_meaning_dec_layers, n_doha_dec_layers, d_ff, dropout, max_len, pad_id):
        super().__init__()
        self.pad_id = pad_id; self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_enc = PositionalEncoding(d_model, max_len, dropout)
        enc_l = nn.TransformerEncoderLayer(d_model, n_heads, d_ff, dropout, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_l, num_layers=n_enc_layers)
        m_dec_l = nn.TransformerDecoderLayer(d_model, n_heads, d_ff, dropout, batch_first=True, norm_first=True)
        self.meaning_decoder = nn.TransformerDecoder(m_dec_l, num_layers=n_meaning_dec_layers)
        self.doha_dec_layers = nn.ModuleList([DohaDecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_doha_dec_layers)])
        self.doha_dec_norm = nn.LayerNorm(d_model)
        self.meaning_proj = nn.Linear(d_model, vocab_size, bias=False)
        self.doha_proj = nn.Linear(d_model, vocab_size, bias=False)
        self.meaning_proj.weight = self.embedding.weight
        self.doha_proj.weight = self.embedding.weight

    def encode(self, src, src_mask):
        x = self.pos_enc(self.embedding(src) * math.sqrt(self.d_model))
        return self.encoder(x, src_key_padding_mask=~src_mask)

    def decode_meaning(self, tgt, memory, src_mask):
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1)).to(tgt.device)
        x = self.pos_enc(self.embedding(tgt) * math.sqrt(self.d_model))
        return self.meaning_decoder(x, memory, tgt_mask=tgt_mask, memory_key_padding_mask=~src_mask)

    def decode_doha(self, tgt, enc_mem, mean_mem, enc_mask, mean_mask):
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1)).to(tgt.device)
        x = self.pos_enc(self.embedding(tgt) * math.sqrt(self.d_model))
        for layer in self.doha_dec_layers:
            x = layer(x, enc_mem, mean_mem, tgt_mask, ~enc_mask, ~mean_mask)
        return self.doha_dec_norm(x)

# ==========================================
# 3. DOWNLOAD & LOAD (डाउनलोड और लोड)
# ==========================================
print("--- Downloading Stage 2 v3 Model ---")
MODEL_PATH = hf_hub_download(repo_id=HF_REPO_ID, filename="best_model.pt")
TOKENIZER_PATH = hf_hub_download(repo_id=HF_REPO_ID, filename="tokenizer.model")

sp = spm.SentencePieceProcessor()
sp.load(TOKENIZER_PATH)
VOCAB_SIZE = sp.get_piece_size()
DANDAA_ID  = sp.piece_to_id('॥')
STOP_TOKENS = {cfg.EOS_ID, DANDAA_ID}

model = UnifiedDohaModel(VOCAB_SIZE, cfg.D_MODEL, cfg.N_HEADS, cfg.N_ENC_LAYERS, 
                         cfg.N_MEANING_DEC_LAYERS, cfg.N_DOHA_DEC_LAYERS, cfg.D_FF, 
                         cfg.DROPOUT, cfg.MAX_SEQ_LEN, cfg.PAD_ID).to(DEVICE)

ckpt = torch.load(MODEL_PATH, map_location=DEVICE)
# DataParallel handle
state_dict = {k.replace('module.', ''): v for k, v in ckpt['model_state'].items()}
model.load_state_dict(state_dict)
model.eval()
print(f"--- Model Loaded (Vocab: {VOCAB_SIZE}) ---")

# ==========================================
# 4. SAMPLING & GENERATION LOGIC
# ==========================================
def top_k_top_p_sample(logits, temperature=0.8, top_k=50, top_p=0.92, past_ids=None, rep_penalty=1.3):
    logits = logits.squeeze(0).float() / temperature
    if past_ids and rep_penalty > 1.0:
        for tid in set(past_ids[-20:]):
            logits[tid] /= rep_penalty if logits[tid] > 0 else (1/rep_penalty)
    if top_k > 0:
        v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
        logits[logits < v[-1]] = float('-inf')
    if top_p < 1.0:
        p_sort, p_idx = torch.sort(F.softmax(logits, dim=-1), descending=True)
        cumsum = torch.cumsum(p_sort, dim=-1)
        p_sort[(cumsum - p_sort) > top_p] = 0.0
        next_token = torch.multinomial(p_sort, 1)
        return p_idx.gather(-1, next_token)
    return torch.multinomial(F.softmax(logits, dim=-1), 1)

@torch.no_grad()
def generate(model, sp, theme, context):
    # Phase 1: Meaning
    enc_text = f"<theme> {theme} </theme> <context> {context} </context>"
    enc_ids = torch.tensor([sp.encode(enc_text)], dtype=torch.long).to(DEVICE)
    enc_mask = (enc_ids != cfg.PAD_ID)
    enc_mem = model.encode(enc_ids, enc_mask)
    
    mean_ids = [cfg.BOS_ID]
    for _ in range(cfg.MAX_MEANING_LEN):
        m_in = torch.tensor([mean_ids], dtype=torch.long).to(DEVICE)
        m_out = model.decode_meaning(m_in, enc_mem, enc_mask)
        next_id = top_k_top_p_sample(model.meaning_proj(m_out[:, -1, :]), 
                                     past_ids=mean_ids, rep_penalty=cfg.GEN_REP_PENALTY).item()
        if next_id == cfg.EOS_ID: break
        mean_ids.append(next_id)
    gen_meaning = sp.decode(mean_ids[1:])
    
    # Phase 2: Doha
    enc_text_full = f"{enc_text} <meaning> {gen_meaning} </meaning>"
    enc_ids_full = torch.tensor([sp.encode(enc_text_full)], dtype=torch.long).to(DEVICE)
    enc_mask_full = (enc_ids_full != cfg.PAD_ID)
    enc_mem_full = model.encode(enc_ids_full, enc_mask_full)
    
    m_mem = model.decode_meaning(torch.tensor([mean_ids], dtype=torch.long).to(DEVICE), 
                                  enc_mem_full, enc_mask_full)
    m_mask = (torch.tensor([mean_ids]) != cfg.PAD_ID).to(DEVICE)
    
    doha_ids = [cfg.BOS_ID]
    for _ in range(cfg.MAX_DOHA_LEN):
        d_in = torch.tensor([doha_ids], dtype=torch.long).to(DEVICE)
        d_out = model.decode_doha(d_in, enc_mem_full, m_mem, enc_mask_full, m_mask)
        next_id = top_k_top_p_sample(model.doha_proj(d_out[:, -1, :]), 
                                     past_ids=doha_ids, rep_penalty=cfg.GEN_DOHA_REP_PEN).item()
        doha_ids.append(next_id)
        if next_id in STOP_TOKENS: break
        
    return gen_meaning, sp.decode(doha_ids[1:])
# ── Matra counting ──────────────────────────────────────────
def count_matras(text):
    """Count matras in a Hindi text string."""
    # Long vowels — 2 matras
    long_vowels = set('आईऊएऐओऔाीूेैोौ')
    # Short vowels — 1 matra
    short_vowels = set('अइउऋिुृ')
    # Anusvara, visarga — 1 matra
    other_matras = set('ंःँ')

    count = 0
    for ch in text:
        if ch in long_vowels:
            count += 2
        elif ch in short_vowels or ch in other_matras:
            count += 1
        elif '\u0900' <= ch <= '\u097F':
            # Any other Devanagari char (consonants etc) = 1 matra
            count += 1
    return count


def matra_score(doha_text):
    """
    Returns (penalty, line1_matras, line2_matras).
    penalty=0 means perfect 24 matras per line.
    Lower penalty = better.
    """
    # Clean and split into lines
    text = doha_text.replace('॥', '').strip()
    # Try newline split first, then comma
    if '\n' in text:
        lines = [l.strip() for l in text.split('\n') if l.strip()]
    elif ',' in text:
        lines = [l.strip() for l in text.split(',') if l.strip()]
    else:
        lines = [text]

    if len(lines) < 2:
        return 999, 0, 0

    l1 = count_matras(lines[0])
    l2 = count_matras(lines[1])
    penalty = abs(l1 - 24) + abs(l2 - 24)
    return penalty, l1, l2


def generate_doha_best_of_n(model, sp, theme, context, n=5,
                              temperature=None, top_k=None, top_p=None):
    """
    Generate n dohas and return the one with best matra count.
    Target: 24 matras per line.
    """
    candidates = []
    for i in range(n):
        meaning, doha = generate_doha(
            model, sp, theme, context,
            temperature=temperature, top_k=top_k, top_p=top_p,
            verbose=False
        )
        penalty, l1, l2 = matra_score(doha)
        candidates.append({
            'doha'     : doha,
            'meaning'  : meaning,
            'penalty'  : penalty,
            'l1_matras': l1,
            'l2_matras': l2,
        })

    # Sort by matra penalty — lowest is best
    candidates.sort(key=lambda x: x['penalty'])
    best = candidates[0]

    print(f"थीम    : {theme}")
    print(f"संदर्भ : {context}")
    print(f"अर्थ   : {best['meaning']}")
    print(f"दोहा   : {best['doha']}")
    print(f"मात्रा : पंक्ति1={best['l1_matras']}/24  पंक्ति2={best['l2_matras']}/24  "
          f"दंड={best['penalty']}")
    print(f"सभी {n} प्रयास:")
    for j, c in enumerate(candidates):
        marker = '★' if j == 0 else ' '
        print(f"  {marker} [{j+1}] penalty={c['penalty']} "
              f"({c['l1_matras']}/{c['l2_matras']}) : {c['doha'][:60]}")
    print("-" * 60)

    return best


print("Matra scoring + best-of-N generation ready ✅")
# ==========================================
# 5. RUN BATCH TESTS (बैच टेस्ट)
# ==========================================
ntc = [
    # --- जीवन और दर्शन (Life & Philosophy) ---
    {'theme': 'संघर्ष', 'context': 'पत्थर का चोट सहकर मूरत बनना'},
    {'theme': 'त्याग', 'context': 'दीपक का खुद जलकर दूसरों को प्रकाश देना'},
    {'theme': 'बदलाव', 'context': 'पुरानी केंचुली छोड़ता हुआ सर्प'},
    {'theme': 'नश्वरता', 'context': 'पानी के बुलबुले का क्षण भर में मिट जाना'},
    {'theme': 'शांति', 'context': 'भीड़भाड़ के बीच मन का मौन हो जाना'},
    {'theme': 'अनुभव', 'context': 'सफेद बालों में छिपे जीवन के गहरे सबक'},
    {'theme': 'भाग्य', 'context': 'समुद्र किनारे लहरों का इंतज़ार करता नाविक'},
    {'theme': 'अस्तित्व', 'context': 'रेत पर बने पैरों के निशान और आती लहरें'},
    {'theme': 'शून्यता', 'context': 'रात के सन्नाटे में ब्रह्मांड की गूँज'},
    {'theme': 'संतुलन', 'context': 'साइकिल के पहिये का निरंतर घूमते रहना'},

    # --- प्रकृति और प्रतीक (Nature & Symbols) ---
    {'theme': 'आशा', 'context': 'रेगिस्तान के बीचों-बीच खिलता हुआ एक फूल'},
    {'theme': 'उदारता', 'context': 'बादलों का बिना भेदभाव के बंजर पर बरसना'},
    {'theme': 'तटस्थता', 'context': 'कमल के पत्ते पर पानी की बूंद का न ठहरना'},
    {'theme': 'दृढ़ता', 'context': 'तूफान में भी अडिग खड़ा हिमालय'},
    {'theme': 'मर्यादा', 'context': 'समुद्र का अपनी सीमाओं को कभी न लांघना'},
    {'theme': 'स्वतंत्रता', 'context': 'पिंजरा तोड़कर खुले आकाश में उड़ता पक्षी'},
    {'theme': 'धैर्य', 'context': 'पतझड़ के बाद नई कोंपलों का धैर्यपूर्ण इंतज़ार'},
    {'theme': 'पुनर्जन्म', 'context': 'राख से उठती हुई फीनिक्स की उड़ान'},
    {'theme': 'शक्ति', 'context': 'चट्टानों का सीना चीरकर बहता झरना'},
    {'theme': 'अनंत', 'context': 'क्षितिज जहाँ धरती और आकाश मिलते प्रतीत होते हैं'},

    # --- मानवीय भावनाएँ (Human Emotions) ---
    {'theme': 'ममता', 'context': 'चिड़िया का अपने बच्चों के लिए तिनका-तिनका जोड़ना'},
    {'theme': 'क्रोध', 'context': 'सुलगता हुआ कोयला जो खुद को पहले जलाता है'},
    {'theme': 'ईर्ष्या', 'context': 'दीमक की तरह अंदर ही अंदर खोखला करना'},
    {'theme': 'करुणा', 'context': 'घायल पक्षी को गोद में उठाकर मरहम लगाना'},
    {'theme': 'उत्साह', 'context': 'भोर की पहली किरण के साथ नई शुरुआत'},
    {'theme': 'पछतावा', 'context': 'चिड़िया के खेत चुग जाने के बाद रखवाली करना'},
    {'theme': 'भय', 'context': 'अंधेरी रात में अपनी ही परछाईं से डरना'},
    {'theme': 'प्रेम', 'context': 'चाँदनी का रात से बिना शर्त जुड़ाव'},
    {'theme': 'एकाकीपन', 'context': 'भरी महफिल में खुद को अकेला पाना'},
    {'theme': 'विस्मय', 'context': 'छोटे शिशु की आँखों में दुनिया का कौतूहल'},

    # --- नैतिक मूल्य (Ethics & Values) ---
    {'theme': 'विनम्रता', 'context': 'फलों से लदी टहनियों का जमीन की ओर झुक जाना'},
    {'theme': 'क्षमा', 'context': 'चंदन के वृक्ष का कुल्हाड़ी को भी सुगंधित करना'},
    {'theme': 'कृतज्ञता', 'context': 'धूप में छाया देने वाले वृक्ष को न काटना'},
    {'theme': 'ईमानदारी', 'context': 'अंधेरे में भी सही रास्ते पर चलने का साहस'},
    {'theme': 'कर्तव्य', 'context': 'सीमा पर डटे सैनिक की अडिग निष्ठा'},
    {'theme': 'संयम', 'context': 'उफनती नदी का तटों की मर्यादा में रहना'},
    {'theme': 'एकता', 'context': 'तिनकों से बनी रस्सी का हाथी को बांध लेना'},
    {'theme': 'सत्य', 'context': 'बादलों की ओट में छिपा सूर्य जो देर-सबेर निकलेगा'},
    {'theme': 'परोपकार', 'context': 'मधुमक्खी का दूसरों के लिए शहद इकठ्ठा करना'},
    {'theme': 'श्रद्धा', 'context': 'पत्थर की मूरत में भी ईश्वर का वास देखना'},

    # --- चुनौतियाँ और सफलता (Challenges & Success) ---
    {'theme': 'परिश्रम', 'context': 'चींटी का अपने वजन से भारी दाना पहाड़ पर ले जाना'},
    {'theme': 'एकाग्रता', 'context': 'अर्जुन की दृष्टि और केवल मछली की आँख'},
    {'theme': 'लक्ष्य', 'context': 'बाण का धनुष से छूटकर निशाने की ओर बढ़ना'},
    {'theme': 'साधना', 'context': 'बूँद-बूँद गिरने से कठोर पत्थर पर पड़ा निशान'},
    {'theme': 'जीत', 'context': 'हजार बार गिरने के बाद एक बार फिर खड़े होना'},
    {'theme': 'अवसर', 'context': 'दस्तक देती हवा जिसे पहचानना जरूरी है'},
    {'theme': 'रणनीति', 'context': 'बिसात पर चाल चलने से पहले का मौन चिंतन'},
    {'theme': 'दृष्टिकोण', 'context': 'आधे भरे गिलास को पूरा देखना'},
    {'theme': 'प्रतिभा', 'context': 'कीचड़ के बीच भी अपनी आभा बिखेरता कमल'},
    {'theme': 'अनुशासन', 'context': 'सेना की तालबद्ध कदमताल'},

    # --- रिश्ते और समाज (Relationships & Society) ---
    {'theme': 'दोस्ती', 'context': 'कृष्ण और सुदामा के कच्चे चावलों की पोटली'},
    {'theme': 'विश्वास', 'context': 'अंधेरे में भी पिता का हाथ थामे चलता बच्चा'},
    {'theme': 'संस्कार', 'context': 'जड़ों की मजबूती जो पेड़ को गिरने नहीं देती'},
    {'theme': 'लोभ', 'context': 'सोने के अंडे देने वाली मुर्गी को मार देना'},
    {'theme': 'स्वार्थ', 'context': 'काम निकलने पर साये का भी साथ छोड़ देना'},
    {'theme': 'सम्मान', 'context': 'बुजुर्गों की झुर्रियों में छिपे ज्ञान का आदर'},
    {'theme': 'विवाद', 'context': 'एक ही बात के दो अलग-अलग छोरों पर खड़े होना'},
    {'theme': 'परंपरा', 'context': 'मिट्टी के दीये की लौ जो पीढ़ी-दर-पीढ़ी जलती है'},
    {'theme': 'घृणा', 'context': 'हाथ में कीचड़ लेकर दूसरों पर फेंकने की कोशिश'},
    {'theme': 'अतिथि', 'context': 'द्वार पर आए अंजान में देवत्व की खोज'},

    # --- ज्ञान और बुद्धि (Knowledge & Wisdom) ---
    {'theme': 'जिज्ञासा', 'context': 'पत्ते के गिरने के पीछे का "क्यों" ढूँढना'},
    {'theme': 'अज्ञान', 'context': 'दीये के नीचे का घना अंधेरा'},
    {'theme': 'विवेक', 'context': 'हंस का दूध और पानी को अलग कर देना'},
    {'theme': 'सादगी', 'context': 'मिट्टी के घड़े के शीतल जल का आनंद'},
    {'theme': 'स्मृति', 'context': 'पुरानी धूल भरी तस्वीरों में कैद मुस्कान'},
    {'theme': 'भ्रम', 'context': 'रेगिस्तान में मृगजल की अंतहीन तलाश'},
    {'theme': 'मौन', 'context': 'शब्दों के समाप्त होने पर सत्य का उदय'},
    {'theme': 'साक्षरता', 'context': 'काले अक्षरों में छिपे संसार की चाबी पाना'},
    {'theme': 'तर्क', 'context': 'अंधेरे में मशाल जलाकर गड्ढे को देखना'},
    {'theme': 'प्रेरणा', 'context': 'बुझते दीये को दूसरी लौ से प्रज्वलित करना'},

    # --- समय और गति (Time & Motion) ---
    {'theme': 'प्रतीक्षा', 'context': 'शबरी के मीठे बेर और राम का इंतज़ार'},
    {'theme': 'निरंतरता', 'context': 'घड़ी की टिक-टिक जो कभी नहीं थकती'},
    {'theme': 'विश्राम', 'context': 'दिन भर की थकान के बाद माँ की गोद'},
    {'theme': 'बचपन', 'context': 'कागज की नाव और बारिश का पानी'},
    {'theme': 'बुढ़ापा', 'context': 'ढलते सूरज की शांत और सौम्य लालिमा'},
    {'theme': 'युवावस्था', 'context': 'उफनती लहरों की असीम ऊर्जा'},
    {'theme': 'कल', 'context': 'एक ऐसा सपना जो कभी हाथ नहीं आता'},
    {'theme': 'आज', 'context': 'मुट्ठी में बंद रेत जो धीरे-धीरे फिसल रही है'},
    {'theme': 'क्षण', 'context': 'कैमरे की एक क्लिक में कैद पूरी जिंदगी'},
    {'theme': 'इतिहास', 'context': 'खंडहरों की दीवारों पर लिखी वीरगाथाएं'},

    # --- अन्य महत्वपूर्ण विषय (Miscellaneous) ---
    {'theme': 'खोज', 'context': 'अंधेरी गुफा में मशाल लेकर हीरे की तलाश'},
    {'theme': 'ममता', 'context': 'गाय का अपने बछड़े को चाटना'},
    {'theme': 'लचीलापन', 'context': 'हवा के झोंके में झुकती घास जो टूटती नहीं'},
    {'theme': 'मजबूरी', 'context': 'पिंजरे में बंद शेर का बेबस दहाड़ना'},
    {'theme': 'उत्सव', 'context': 'अमावस की रात में दीपों की कतार'},
    {'theme': 'अन्याय', 'context': 'ताकतवर भेड़िये का मेमने पर प्रहार'},
    {'theme': 'न्याय', 'context': 'तराजू के दोनों पलड़ों का बराबर होना'},
    {'theme': 'कला', 'context': 'कोरे कागज पर रंगों का जीवंत संवाद'},
    {'theme': 'साहस', 'context': 'अंधेरी सुरंग के अंत में प्रकाश की उम्मीद'},
    {'theme': 'निर्मलता', 'context': 'पहाड़ों से निकलता कांच जैसा साफ पानी'},
    {'theme': 'वफादारी', 'context': 'मालिक की चौखट पर बैठा बूढ़ा कुत्ता'},
    {'theme': 'गर्व', 'context': 'तिरंगे का आकाश की ऊंचाइयों में लहराना'},
    {'theme': 'सादगी', 'context': 'बिना गहनों के भी चेहरे का प्राकृतिक तेज'},
    {'theme': 'उलझन', 'context': 'धागे का ऐसा सिरा जो कहीं खो गया है'},
    {'theme': 'आकर्षण', 'context': 'पतंगे का जलती शमशाम की ओर खिंचे जाना'},
    {'theme': 'वैराग्य', 'context': 'राजपाट छोड़कर सिद्धार्थ का वन को जाना'},
    {'theme': 'अभिमान', 'context': 'रावण के दस सिरों का अपने ज्ञान पर गुमान'},
    {'theme': 'सुख', 'context': 'गर्मियों की दोपहर में घने पेड़ की छाँव'},
    {'theme': 'दुःख', 'context': 'सूखी आँखों से बहते अदृश्य आँसू'},
    {'theme': 'ईश्वर', 'context': 'हवा की तरह जो दिखती नहीं पर महसूस होती है'},

    # (इसी क्रम में अगले 100 विषय...)
    {'theme': 'मौन', 'context': 'शब्दों के थकने के बाद शुरू होने वाली बातचीत'},
    {'theme': 'क्रूरता', 'context': 'सूखे पेड़ को जड़ से उखाड़ देना'},
    {'theme': 'सुगंध', 'context': 'मिट्टी पर पहली बारिश की बूंदें'},
    {'theme': 'साक्षी', 'context': 'पुराना बरगद जिसने सदियां गुजरते देखीं'},
    {'theme': 'संगीत', 'context': 'बाँसुरी के छेदों से निकलता रूहानी स्वर'},
    {'theme': 'साहचर्य', 'context': 'धूप और छाँव का लुका-छिपी का खेल'},
    {'theme': 'कल्पना', 'context': 'बादलों में आकृतियां ढूँढता एक कवि'},
    {'theme': 'मर्यादा', 'context': 'लक्ष्मण रेखा जो सुरक्षा का घेरा है'},
    {'theme': 'तपस्या', 'context': 'बर्फानी चोटियों पर बिना वस्त्रों के साधु'},
    {'theme': 'विनाश', 'context': 'ज्वालामुखी का दहकता हुआ लावा'},
    {'theme': 'निर्माण', 'context': 'खंडहरों के बीच से नए शहर का उदय'},
    {'theme': 'प्रसिद्धि', 'context': 'आकाश में टूटते तारे की एक क्षण की चमक'},
    {'theme': 'गुमनामी', 'context': 'जंगल में खिले फूल की अनसुनी खुशबू'},
    {'theme': 'अपेक्षा', 'context': 'सूखे खेत का आसमान की ओर ताकना'},
    {'theme': 'संतुष्टि', 'context': 'भरपेट भोजन के बाद एक लंबी डकार'},
    {'theme': 'स्वप्न', 'context': 'बंद आँखों से सात समंदर पार की सैर'},
    {'theme': 'हकीकत', 'context': 'धूप में जलते नंगे पैरों के छाले'},
    {'theme': 'अधिकार', 'context': 'सिंहासन पर बैठते ही बदलती मानसिकता'},
    {'theme': 'अपराध', 'context': 'रात के अंधेरे में चेहरे पर लगा नकाब'},
    {'theme': 'भक्ति', 'context': 'हनुमान का सीना चीरकर राम का नाम दिखाना'},
    {'theme': 'बलिदान', 'context': 'देश के लिए हँसते-हँसते फांसी चढ़ना'},
    {'theme': 'चपलता', 'context': 'गिलहरी का पेड़ पर चढ़ना और उतरना'},
    {'theme': 'गंभीरता', 'context': 'गहरे पानी का शांत व्यवहार'},
    {'theme': 'संशय', 'context': 'दो रास्तों के बीच खड़ा दुविधा में फँसा राही'},
    {'theme': 'निर्भयता', 'context': 'शेर की मांद में घुसकर उसे चुनौती देना'},
    {'theme': 'आलस्य', 'context': 'दीवार पर टंगी रुकी हुई घड़ी'},
    {'theme': 'परिवर्तन', 'context': 'मौसम के साथ बदलता जंगलों का रंग'},
    {'theme': 'एकनिष्ठता', 'context': 'ध्रुव तारे का अपनी जगह पर अडिग रहना'},
    {'theme': 'लालच', 'context': 'मछली का कांटे में फँसे मांस के टुकड़े को निगलना'},
    {'theme': 'क्रान्ति', 'context': 'दबे हुए अंगारों का अचानक भभक उठना'},
    {'theme': 'सहनशीलता', 'context': 'धरती जो सबका बोझ सहती है'},
    {'theme': 'मिठास', 'context': 'गन्ने के रस का अंतिम बूंद तक मीठा होना'},
    {'theme': 'कठोरता', 'context': 'नारियल का बाहरी सख्त कवच'},
    {'theme': 'कोमलता', 'context': 'गुलाब की पंखुड़ियों पर ओस की बूंद'},
    {'theme': 'विछोह', 'context': 'रेलवे स्टेशन पर छूटता हुआ हाथ'},
    {'theme': 'मिलन', 'context': 'नदी का सागर की बाहों में समा जाना'},
    {'theme': 'चोरी', 'context': 'चाँद का सूरज की रोशनी चुराना'},
    {'theme': 'दान', 'context': 'कर्ण का अपने कवच और कुंडल उतार देना'},
    {'theme': 'धोखा', 'context': 'दूध में पानी का सफेद झूठ'},
    {'theme': 'अमृत', 'context': 'ज्ञान के मंथन से निकला निष्कर्ष'},
    {'theme': 'विष', 'context': 'कानों में घोली गई कड़वी बातें'},
    {'theme': 'वैभव', 'context': 'इंद्रधनुष के सात रंगों की सजावट'},
    {'theme': 'दरिद्रता', 'context': 'खाली थाली में चाँद का अक्स देखना'},
    {'theme': 'श्रृंगार', 'context': 'धरती पर फैली मखमली हरी घास'},
    {'theme': 'वैराग्य', 'context': 'श्मशान की राख और जीवन का सत्य'},
    {'theme': 'चेतना', 'context': 'पत्थर के भीतर छिपी हुई आग'},
    {'theme': 'विद्रोही', 'context': 'पिंजरे की तीलियां तोड़ता परिंदा'},
    {'theme': 'शरण', 'context': 'थके हुए राही के लिए सराय का दीया'},
    {'theme': 'उदारता', 'context': 'फलों का स्वयं न खाकर दूसरों को देना'},
    {'theme': 'अहंकार', 'context': 'रावण के सोने की लंका का दहन'},
    {'theme': 'ममता', 'context': 'यशौदा का कान्हा को माखन खिलाना'},
    {'theme': 'विरह', 'context': 'यक्ष की मेघदूत के माध्यम से पुकार'},
    {'theme': 'संदेह', 'context': 'दूध का जला मट्ठा भी फूंककर पीना'},
    {'theme': 'शांति', 'context': 'बुद्ध के चेहरे की सौम्य मुस्कान'},
    {'theme': 'संसार', 'context': 'एक बड़ा मेला जहाँ सब अजनबी हैं'},
    {'theme': 'मर्यादा', 'context': 'नदी का अपने किनारे न तोड़ना'},
    {'theme': 'विजेता', 'context': 'हार मानकर भी फिर से उठने वाला'},
    {'theme': 'पराजय', 'context': 'मैदान छोड़कर भागने का डर'},
    {'theme': 'साहस', 'context': 'अकेले ही भीड़ के खिलाफ खड़े होना'},
    {'theme': 'प्रकृति', 'context': 'पहाड़ों की चोटियों पर ढलती शाम'},
    {'theme': 'मानवता', 'context': 'दुश्मन को भी प्यास लगने पर पानी पिलाना'},
    {'theme': 'सहनशक्ति', 'context': 'लोहे का गर्म होकर हथौड़े की मार सहना'},
    {'theme': 'परिवेश', 'context': 'गाँव की चौपाल और नीम का पेड़'},
    {'theme': 'स्मृति', 'context': 'हवा में तैरती बचपन की खुशबू'},
    {'theme': 'उम्मीद', 'context': 'अंधेरे घर में जलता हुआ नन्हा दीया'},
    {'theme': 'विश्वासघात', 'context': 'आस्तीन में पाला हुआ सांप'},
    {'theme': 'करुणा', 'context': 'रास्ते के पत्थर को हटाना ताकि कोई गिरे नहीं'},
    {'theme': 'सौन्दर्य', 'context': 'पूर्णमासी का पूरा खिला हुआ चाँद'},
    {'theme': 'कष्ट', 'context': 'नंगे पैर कँटीली राहों पर चलना'},
    {'theme': 'सुविधा', 'context': 'सोने के पिंजरे में बंद पक्षी'},
    {'theme': 'अतृप्ति', 'context': 'रेगिस्तान में भटकता हुआ प्यासा'},
    {'theme': 'तृप्ति', 'context': 'मनचाहा वरदान मिलने की खुशी'},
    {'theme': 'अंतर्मन', 'context': 'दर्पण जो चेहरा नहीं चरित्र दिखाता है'},
    {'theme': 'बाहरी दिखावा', 'context': 'ऊपर से फिट और अंदर से अनफिट'},
    {'theme': 'साधुता', 'context': 'बिना किसी मोह के दुनिया से गुजर जाना'},
    {'theme': 'कुटिलता', 'context': 'शकुनि के पासे और बिछाई गई बिसात'},
    {'theme': 'चमक', 'context': 'अंधेरे में जुगनू की छोटी सी रोशनी'},
    {'theme': 'गहराई', 'context': 'समुद्र के तल में छिपे हुए मोती'},
    {'theme': 'ऊंचाई', 'context': 'ईगल की नजर जो बादलों के ऊपर है'},
    {'theme': 'विस्तार', 'context': 'फैला हुआ नीला अंबर जिसका अंत नहीं'},
    {'theme': 'संकीर्णता', 'context': 'कुएँ का मेंढक जो उसे ही दुनिया मानता है'},
    {'theme': 'स्थिरता', 'context': 'झील का पानी जिसमें लहरें नहीं हैं'},
    {'theme': 'चंचलता', 'context': 'पवन का झोंका जो यहाँ-वहाँ भटकता है'},
    {'theme': 'सृष्टि', 'context': 'मिट्टी से बने खिलौने और उनका रचयिता'},
    {'theme': 'प्रलय', 'context': 'शिव का तांडव और टूटता ब्रह्मांड'},
    {'theme': 'आस्था', 'context': 'कांच की दीवार के पार भी रास्ता देखना'},
    {'theme': 'तर्कशक्ति', 'context': 'बीरबल की खिचड़ी और उसकी बुद्धिमत्ता'},
    {'theme': 'लालसा', 'context': 'इंद्रधनुष को मुट्ठी में पकड़ने की कोशिश'},
    {'theme': 'निर्लेपता', 'context': 'संसार में रहकर भी संसार का न होना'},
    {'theme': 'विद्रूपता', 'context': 'सुंदर चेहरे के पीछे छिपा बदसूरत विचार'},
    {'theme': 'समर्पण', 'context': 'बाँसुरी का स्वयं को खाली कर देना'},
    {'theme': 'आकर्षण', 'context': 'लोहे का चुंबक की ओर खिंचना'},
    {'theme': 'प्रतिशोध', 'context': 'जंगल की आग जो सब भस्म कर दे'},
    {'theme': 'सत्यनिष्ठा', 'context': 'हरिश्चंद्र का श्मशान में कर मांगना'},
    {'theme': 'धैर्य', 'context': 'बर्फ का पिघलकर धीरे-धीरे पानी बनना'},
    {'theme': 'उत्सुकता', 'context': 'बंद लिफाफे में क्या है यह जानने की बेचैनी'},
    {'theme': 'अहंकार', 'context': 'पहाड़ का अपनी ऊंचाई पर इतराना'},
    {'theme': 'विनम्रता', 'context': 'नदी का सागर में मिलकर अपना अस्तित्व खोना'},
    {'theme': 'जीवन', 'context': 'सांसों की एक ऐसी माला जो कभी टूटती है'},
    {'theme': 'मृत्यु', 'context': 'एक लंबी नींद जिसके बाद कोई नहीं जागता'}
]

print("\n" + "="*60)
print(" RUNNING STAGE 2 v3 BATCH TESTS ")
print("="*60)

for i, test in enumerate(ntc):
    meaning, doha = generate(model, sp, test['theme'], test['context'])
    print(f"\nTest {i+1}:")
    print(f"Theme   : {test['theme']}")
    print(f"Context : {test['context']}")
    print(f"Meaning : {meaning}")
    print(f"Doha    : {doha}")
    print("-" * 40)

print("\nBatch testing complete. ✅")

--- Downloading Stage 2 v3 Model ---


/tmp/ipykernel_55/495589005.py:84: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_l, num_layers=n_enc_layers)


--- Model Loaded (Vocab: 8000) ---
Matra scoring + best-of-N generation ready ✅

 RUNNING STAGE 2 v3 BATCH TESTS 

Test 1:
Theme   : संघर्ष
Context : पत्थर का चोट सहकर मूरत बनना
Meaning : कवि कहता है कि कैसे पानी भी जल से भरे हो सकती है, यदि पत्थर के समान ही होता है तो केवल अपनी आँखों को नहीं देखती।
Doha    : कैसे जल से नीर का, कैसे होते नीर। पत्थर ही भी कभी, देखे आँखों के नैन ॥
----------------------------------------

Test 2:
Theme   : त्याग
Context : दीपक का खुद जलकर दूसरों को प्रकाश देना
Meaning : दीपक की आग को जलाकर दूसरों के भीतर से बाहर निकल गए हैं, पर दीपक को स्वयं जलकर उसे जल कर कर जलाये।
Doha    : दीप जले दीपक जला, बाहर को देये आग। दीप सभी जल कर, जला दिया जल नीर ॥
----------------------------------------

Test 3:
Theme   : बदलाव
Context : पुरानी केंचुली छोड़ता हुआ सर्प
Meaning : पुरानी केंद्रों के लिए सुंदर चित्रित हुआ है, जो अपनी पहचान में व्याप्त रहती है और उसकी पुरानी केंद्रों की भाँति अडिगग करते हुए भी अपनी पहचानें।
Doha    : पुरानी केंद्रों के लिए, करती है चित्र। इसीलिए 

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import sentencepiece as spm
import math
import os
from huggingface_hub import hf_hub_download

# ==========================================
# 1. CONFIGURATION (कॉन्फ़िगरेशन)
# ==========================================
class Config:
    D_MODEL              = 256
    N_HEADS              = 8
    N_ENC_LAYERS         = 4
    N_MEANING_DEC_LAYERS = 4
    N_DOHA_DEC_LAYERS    = 4
    D_FF                 = 1024
    DROPOUT              = 0.15
    MAX_SEQ_LEN          = 256
    MAX_MEANING_LEN      = 60
    MAX_DOHA_LEN         = 48
    PAD_ID               = 0
    BOS_ID               = 2
    EOS_ID               = 3
    # Generation Params
    GEN_TEMPERATURE      = 0.8
    GEN_TOP_K            = 50
    GEN_TOP_P            = 0.92
    GEN_REP_PENALTY      = 1.3
    GEN_DOHA_REP_PEN     = 1.5

cfg = Config()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HF_REPO_ID = "nikpatidar333/doha-generation-model_v2" # Stage 2 v3 Repo

# ==========================================
# 2. MODEL ARCHITECTURE (मॉडल संरचना)
# ==========================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

class DohaDecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model); self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model); self.norm4 = nn.LayerNorm(d_model)
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.cross_attn_enc = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.cross_attn_meaning = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(d_ff, d_model), nn.Dropout(dropout))
        self.gate = nn.Parameter(torch.tensor(0.5))
    def forward(self, x, encoder_memory, meaning_memory, tgt_mask, enc_key_padding_mask, meaning_key_padding_mask):
        res = x; x = self.norm1(x)
        attn_out, _ = self.self_attn(x, x, x, attn_mask=tgt_mask)
        x = res + attn_out
        res = x; x_norm = self.norm2(x)
        enc_out, _ = self.cross_attn_enc(x_norm, encoder_memory, encoder_memory, key_padding_mask=enc_key_padding_mask)
        meaning_out, _ = self.cross_attn_meaning(x_norm, meaning_memory, meaning_memory, key_padding_mask=meaning_key_padding_mask)
        g = torch.sigmoid(self.gate)
        x = res + g * enc_out + (1.0 - g) * meaning_out
        res = x; x = self.norm3(x)
        x = res + self.ffn(x)
        return self.norm4(x)

class UnifiedDohaModel(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_enc_layers, n_meaning_dec_layers, n_doha_dec_layers, d_ff, dropout, max_len, pad_id):
        super().__init__()
        self.pad_id = pad_id; self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_enc = PositionalEncoding(d_model, max_len, dropout)
        enc_l = nn.TransformerEncoderLayer(d_model, n_heads, d_ff, dropout, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_l, num_layers=n_enc_layers)
        m_dec_l = nn.TransformerDecoderLayer(d_model, n_heads, d_ff, dropout, batch_first=True, norm_first=True)
        self.meaning_decoder = nn.TransformerDecoder(m_dec_l, num_layers=n_meaning_dec_layers)
        self.doha_dec_layers = nn.ModuleList([DohaDecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_doha_dec_layers)])
        self.doha_dec_norm = nn.LayerNorm(d_model)
        self.meaning_proj = nn.Linear(d_model, vocab_size, bias=False)
        self.doha_proj = nn.Linear(d_model, vocab_size, bias=False)
        self.meaning_proj.weight = self.embedding.weight
        self.doha_proj.weight = self.embedding.weight

    def encode(self, src, src_mask):
        x = self.pos_enc(self.embedding(src) * math.sqrt(self.d_model))
        return self.encoder(x, src_key_padding_mask=~src_mask)

    def decode_meaning(self, tgt, memory, src_mask):
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1)).to(tgt.device)
        x = self.pos_enc(self.embedding(tgt) * math.sqrt(self.d_model))
        return self.meaning_decoder(x, memory, tgt_mask=tgt_mask, memory_key_padding_mask=~src_mask)

    def decode_doha(self, tgt, enc_mem, mean_mem, enc_mask, mean_mask):
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1)).to(tgt.device)
        x = self.pos_enc(self.embedding(tgt) * math.sqrt(self.d_model))
        for layer in self.doha_dec_layers:
            x = layer(x, enc_mem, mean_mem, tgt_mask, ~enc_mask, ~mean_mask)
        return self.doha_dec_norm(x)

# ==========================================
# 3. DOWNLOAD & LOAD (डाउनलोड और लोड)
# ==========================================
print("--- Downloading Stage 2 v3 Model ---")
MODEL_PATH = hf_hub_download(repo_id=HF_REPO_ID, filename="best_model.pt")
TOKENIZER_PATH = hf_hub_download(repo_id=HF_REPO_ID, filename="tokenizer.model")

sp = spm.SentencePieceProcessor()
sp.load(TOKENIZER_PATH)
VOCAB_SIZE = sp.get_piece_size()
DANDAA_ID  = sp.piece_to_id('॥')
STOP_TOKENS = {cfg.EOS_ID, DANDAA_ID}

model = UnifiedDohaModel(VOCAB_SIZE, cfg.D_MODEL, cfg.N_HEADS, cfg.N_ENC_LAYERS, 
                         cfg.N_MEANING_DEC_LAYERS, cfg.N_DOHA_DEC_LAYERS, cfg.D_FF, 
                         cfg.DROPOUT, cfg.MAX_SEQ_LEN, cfg.PAD_ID).to(DEVICE)

ckpt = torch.load(MODEL_PATH, map_location=DEVICE)
# DataParallel handle
state_dict = {k.replace('module.', ''): v for k, v in ckpt['model_state'].items()}
model.load_state_dict(state_dict)
model.eval()
print(f"--- Model Loaded (Vocab: {VOCAB_SIZE}) ---")

# ==========================================
# 4. SAMPLING & GENERATION LOGIC
# ==========================================
def top_k_top_p_sample(logits, temperature=0.8, top_k=50, top_p=0.92, past_ids=None, rep_penalty=1.3):
    logits = logits.squeeze(0).float() / temperature
    if past_ids and rep_penalty > 1.0:
        for tid in set(past_ids[-20:]):
            logits[tid] /= rep_penalty if logits[tid] > 0 else (1/rep_penalty)
    if top_k > 0:
        v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
        logits[logits < v[-1]] = float('-inf')
    if top_p < 1.0:
        p_sort, p_idx = torch.sort(F.softmax(logits, dim=-1), descending=True)
        cumsum = torch.cumsum(p_sort, dim=-1)
        p_sort[(cumsum - p_sort) > top_p] = 0.0
        next_token = torch.multinomial(p_sort, 1)
        return p_idx.gather(-1, next_token)
    return torch.multinomial(F.softmax(logits, dim=-1), 1)

@torch.no_grad()
def generate(model, sp, theme, context):
    # Phase 1: Meaning
    enc_text = f"<theme> {theme} </theme> <context> {context} </context>"
    enc_ids = torch.tensor([sp.encode(enc_text)], dtype=torch.long).to(DEVICE)
    enc_mask = (enc_ids != cfg.PAD_ID)
    enc_mem = model.encode(enc_ids, enc_mask)
    
    mean_ids = [cfg.BOS_ID]
    for _ in range(cfg.MAX_MEANING_LEN):
        m_in = torch.tensor([mean_ids], dtype=torch.long).to(DEVICE)
        m_out = model.decode_meaning(m_in, enc_mem, enc_mask)
        next_id = top_k_top_p_sample(model.meaning_proj(m_out[:, -1, :]), 
                                     past_ids=mean_ids, rep_penalty=cfg.GEN_REP_PENALTY).item()
        if next_id == cfg.EOS_ID: break
        mean_ids.append(next_id)
    gen_meaning = sp.decode(mean_ids[1:])
    
    # Phase 2: Doha
    enc_text_full = f"{enc_text} <meaning> {gen_meaning} </meaning>"
    enc_ids_full = torch.tensor([sp.encode(enc_text_full)], dtype=torch.long).to(DEVICE)
    enc_mask_full = (enc_ids_full != cfg.PAD_ID)
    enc_mem_full = model.encode(enc_ids_full, enc_mask_full)
    
    m_mem = model.decode_meaning(torch.tensor([mean_ids], dtype=torch.long).to(DEVICE), 
                                  enc_mem_full, enc_mask_full)
    m_mask = (torch.tensor([mean_ids]) != cfg.PAD_ID).to(DEVICE)
    
    doha_ids = [cfg.BOS_ID]
    for _ in range(cfg.MAX_DOHA_LEN):
        d_in = torch.tensor([doha_ids], dtype=torch.long).to(DEVICE)
        d_out = model.decode_doha(d_in, enc_mem_full, m_mem, enc_mask_full, m_mask)
        next_id = top_k_top_p_sample(model.doha_proj(d_out[:, -1, :]), 
                                     past_ids=doha_ids, rep_penalty=cfg.GEN_DOHA_REP_PEN).item()
        doha_ids.append(next_id)
        if next_id in STOP_TOKENS: break
        
    return gen_meaning, sp.decode(doha_ids[1:])
# ── Matra counting ──────────────────────────────────────────
def count_matras(text):
    """Count matras in a Hindi text string."""
    # Long vowels — 2 matras
    long_vowels = set('आईऊएऐओऔाीूेैोौ')
    # Short vowels — 1 matra
    short_vowels = set('अइउऋिुृ')
    # Anusvara, visarga — 1 matra
    other_matras = set('ंःँ')

    count = 0
    for ch in text:
        if ch in long_vowels:
            count += 2
        elif ch in short_vowels or ch in other_matras:
            count += 1
        elif '\u0900' <= ch <= '\u097F':
            # Any other Devanagari char (consonants etc) = 1 matra
            count += 1
    return count


def matra_score(doha_text):
    """
    Returns (penalty, line1_matras, line2_matras).
    penalty=0 means perfect 24 matras per line.
    Lower penalty = better.
    """
    # Clean and split into lines
    text = doha_text.replace('॥', '').strip()
    # Try newline split first, then comma
    if '\n' in text:
        lines = [l.strip() for l in text.split('\n') if l.strip()]
    elif ',' in text:
        lines = [l.strip() for l in text.split(',') if l.strip()]
    else:
        lines = [text]

    if len(lines) < 2:
        return 999, 0, 0

    l1 = count_matras(lines[0])
    l2 = count_matras(lines[1])
    penalty = abs(l1 - 24) + abs(l2 - 24)
    return penalty, l1, l2


def generate_doha_best_of_n(model, sp, theme, context, n=5,
                              temperature=None, top_k=None, top_p=None):
    """
    Generate n dohas and return the one with best matra count.
    Target: 24 matras per line.
    """
    candidates = []
    for i in range(n):
        meaning, doha = generate_doha(
            model, sp, theme, context,
            temperature=temperature, top_k=top_k, top_p=top_p,
            verbose=False
        )
        penalty, l1, l2 = matra_score(doha)
        candidates.append({
            'doha'     : doha,
            'meaning'  : meaning,
            'penalty'  : penalty,
            'l1_matras': l1,
            'l2_matras': l2,
        })

    # Sort by matra penalty — lowest is best
    candidates.sort(key=lambda x: x['penalty'])
    best = candidates[0]

    print(f"थीम    : {theme}")
    print(f"संदर्भ : {context}")
    print(f"अर्थ   : {best['meaning']}")
    print(f"दोहा   : {best['doha']}")
    print(f"मात्रा : पंक्ति1={best['l1_matras']}/24  पंक्ति2={best['l2_matras']}/24  "
          f"दंड={best['penalty']}")
    print(f"सभी {n} प्रयास:")
    for j, c in enumerate(candidates):
        marker = '★' if j == 0 else ' '
        print(f"  {marker} [{j+1}] penalty={c['penalty']} "
              f"({c['l1_matras']}/{c['l2_matras']}) : {c['doha'][:60]}")
    print("-" * 60)

    return best


print("Matra scoring + best-of-N generation ready ✅")
# ==========================================
# 5. RUN BATCH TESTS (बैच टेस्ट)
# ==========================================
ntc = [
    {'theme': 'प्रेम', 'context': 'माँ का प्यार'},
    {'theme': 'दोस्ती', 'context': 'मुसीबत में मदद'},
    {'theme': 'क्रोध', 'context': 'गुस्से में झगड़ा'},
    {'theme': 'क्षमा', 'context': 'गलती माफ करना'},
    {'theme': 'ईर्ष्या', 'context': 'तरक्की से जलना'},
    {'theme': 'करुणा', 'context': 'बीमार की सेवा'},
    {'theme': 'लोभ', 'context': 'पैसे का लालच'},
    {'theme': 'ममता', 'context': 'बच्चे की फिक्र'},
    {'theme': 'विश्वास', 'context': 'अजनबी पर भरोसा'},
    {'theme': 'धोखा', 'context': 'पीठ पीछे वार'},
    
    {'theme': 'सत्य', 'context': 'हमेशा सच बोलना'},
    {'theme': 'ईमानदारी', 'context': 'गिरा हुआ पर्स लौटाना'},
    {'theme': 'परोपकार', 'context': 'भूखे को खाना'},
    {'theme': 'साहस', 'context': 'डूबते को बचाना'},
    {'theme': 'विनम्रता', 'context': 'बड़ों का आदर'},
    {'theme': 'धैर्य', 'context': 'शांति से इंतज़ार'},
    {'theme': 'संतोष', 'context': 'रूखी-सूखी में खुश'},
    {'theme': 'कर्तव्य', 'context': 'बॉर्डर पर पहरा'},
    {'theme': 'निष्ठा', 'context': 'मालिक का वफादार'},
    {'theme': 'अनुशासन', 'context': 'समय पर उठना'},

    {'theme': 'संघर्ष', 'context': 'कड़ी धूप में मजदूरी'},
    {'theme': 'परिश्रम', 'context': 'खेत में काम'},
    {'theme': 'सफलता', 'context': 'परीक्षा में पास'},
    {'theme': 'असफलता', 'context': 'व्यापार में घाटा'},
    {'theme': 'निराशा', 'context': 'काम ना बनना'},
    {'theme': 'आशा', 'context': 'अच्छे दिन की उम्मीद'},
    {'theme': 'अवसर', 'context': 'नई नौकरी मिलना'},
    {'theme': 'लक्ष्य', 'context': 'डॉक्टर बनने की जिद'},
    {'theme': 'गरीबी', 'context': 'दो वक्त की रोटी'},
    {'theme': 'अमीरी', 'context': 'बड़ा घर और गाड़ी'},

    {'theme': 'एकता', 'context': 'मिलकर काम करना'},
    {'theme': 'भेदभाव', 'context': 'ऊंच-नीच मानना'},
    {'theme': 'शिक्षा', 'context': 'स्कूल जाकर पढ़ना'},
    {'theme': 'अज्ञान', 'context': 'बिना सोचे काम'},
    {'theme': 'अंधविश्वास', 'context': 'बिल्ली का रास्ता काटना'},
    {'theme': 'स्वार्थ', 'context': 'अपना फायदा देखना'},
    {'theme': 'निस्वार्थ', 'context': 'मुफ्त में सेवा'},
    {'theme': 'भ्रष्टाचार', 'context': 'रिश्वत लेना'},
    {'theme': 'न्याय', 'context': 'बेगुनाह को छोड़ना'},
    {'theme': 'अन्याय', 'context': 'गरीब को लूटना'},

    {'theme': 'प्रकृति', 'context': 'बारिश और हरियाली'},
    {'theme': 'समय', 'context': 'बीता हुआ कल'},
    {'theme': 'बदलाव', 'context': 'नई सोच अपनाना'},
    {'theme': 'मृत्यु', 'context': 'बुढ़ापे में मौत'},
    {'theme': 'बचपन', 'context': 'मिट्टी में खेलना'},
    {'theme': 'बुढ़ापा', 'context': 'लाठी का सहारा'},
    {'theme': 'जवानी', 'context': 'नया जोश'},
    {'theme': 'पर्यावरण', 'context': 'पेड़ लगाना'},
    {'theme': 'मौसम', 'context': 'ठंडी हवा'},
    {'theme': 'आपदा', 'context': 'बाढ़ का आना'},

    {'theme': 'शांति', 'context': 'अकेले बैठना'},
    {'theme': 'ध्यान', 'context': 'आँख बंद कर सोचना'},
    {'theme': 'भक्ति', 'context': 'भगवान की पूजा'},
    {'theme': 'पाप', 'context': 'चोरी करना'},
    {'theme': 'पुण्य', 'context': 'प्यासे को पानी'},
    {'theme': 'घमंड', 'context': 'पैसे का गुरूर'},
    {'theme': 'पछतावा', 'context': 'गलती का अहसास'},
    {'theme': 'लालसा', 'context': 'और ज्यादा मांगना'},
    {'theme': 'त्याग', 'context': 'अपना हिस्सा छोड़ना'},
    {'theme': 'मोह', 'context': 'परिवार से लगाव'},

    {'theme': 'निंदा', 'context': 'चुगली करना'},
    {'theme': 'प्रशंसा', 'context': 'तारीफ करना'},
    {'theme': 'सहयोग', 'context': 'काम में हाथ बंटाना'},
    {'theme': 'विवाद', 'context': 'छोटी बात पर बहस'},
    {'theme': 'समझौता', 'context': 'लड़ाई खत्म करना'},
    {'theme': 'जिम्मेदारी', 'context': 'घर चलाना'},
    {'theme': 'लापरवाही', 'context': 'बिना हेलमेट गाड़ी'},
    {'theme': 'सतर्कता', 'context': 'अनजान से दूरी'},
    {'theme': 'मिलावट', 'context': 'दूध में पानी'},
    {'theme': 'प्रेरणा', 'context': 'महान लोगों की कहानी'},

    {'theme': 'डर', 'context': 'अंधेरे से घबराहट'},
    {'theme': 'खुशी', 'context': 'मनपसंद तोहफा'},
    {'theme': 'दुख', 'context': 'प्रियजन की मौत'},
    {'theme': 'अपमान', 'context': 'भरी महफिल में बेइज्जती'},
    {'theme': 'सम्मान', 'context': 'पैर छूकर प्रणाम'},
    {'theme': 'आलस्य', 'context': 'दिन भर सोना'},
    {'theme': 'फुर्ती', 'context': 'जल्दी काम निपटाना'},
    {'theme': 'लालच', 'context': 'मुफ्त का सामान'},
    {'theme': 'दान', 'context': 'अनाथों को कपड़े'},
    {'theme': 'कंजूसी', 'context': 'दवा पर पैसे ना खर्चना'},

    {'theme': 'स्वदेशी', 'context': 'अपने देश का सामान'},
    {'theme': 'देशभक्ति', 'context': 'तिरंगे का सम्मान'},
    {'theme': 'कूटनीति', 'context': 'चालाकी से बात मनवाना'},
    {'theme': 'कला', 'context': 'सुंदर चित्र बनाना'},
    {'theme': 'विज्ञान', 'context': 'नई मशीन बनाना'},
    {'theme': 'परंपरा', 'context': 'दिवाली पर दीये'},
    {'theme': 'आधुनिकता', 'context': 'नए कपड़े पहनना'},
    {'theme': 'स्वास्थ्य', 'context': 'रोज सुबह दौड़ना'},
    {'theme': 'बीमारी', 'context': 'बुखार आना'},
    {'theme': 'थकान', 'context': 'दिन भर का काम'},

    {'theme': 'आराम', 'context': 'छुट्टी के दिन सोना'},
    {'theme': 'यात्रा', 'context': 'पहाड़ों पर घूमना'},
    {'theme': 'इंतजार', 'context': 'स्टेशन पर ट्रेन देखना'},
    {'theme': 'जुदाई', 'context': 'घर से दूर जाना'},
    {'theme': 'मिलन', 'context': 'पुराने दोस्त से मिलना'},
    {'theme': 'यादें', 'context': 'पुरानी फोटो देखना'},
    {'theme': 'समझदारी', 'context': 'सोच-समझकर फैसला'},
    {'theme': 'चापलूसी', 'context': 'बॉस की झूठी तारीफ'},
    {'theme': 'संदेह', 'context': 'चोरी का शक'},
    {'theme': 'मानवता', 'context': 'गिरे हुए को उठाना'}
]

print("\n" + "="*60)
print(" RUNNING STAGE 2 v3 BATCH TESTS ")
print("="*60)

for i, test in enumerate(ntc):
    meaning, doha = generate(model, sp, test['theme'], test['context'])
    print(f"\nTest {i+1}:")
    print(f"Theme   : {test['theme']}")
    print(f"Context : {test['context']}")
    print(f"Meaning : {meaning}")
    print(f"Doha    : {doha}")
    print("-" * 40)

print("\nBatch testing complete. ✅")

--- Downloading Stage 2 v3 Model ---


/tmp/ipykernel_55/2439025636.py:84: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_l, num_layers=n_enc_layers)


--- Model Loaded (Vocab: 8000) ---
Matra scoring + best-of-N generation ready ✅

 RUNNING STAGE 2 v3 BATCH TESTS 

Test 1:
Theme   : प्रेम
Context : माँ का प्यार
Meaning : कवि कहता है कि माँ के प्यार में हमेशा प्रेम के कारण सुख होते हैं, और वह अपनी आँखों में आँसू बहा जाता है। यह माँ की ममता और पवित्रता और प्रेम की भावना को दर्शाता है।
Doha    : प्यार सदा सुख में कभी, मिलता है यार। आँखों में आँसू बसी , बहा जाता नैन ॥
----------------------------------------

Test 2:
Theme   : दोस्ती
Context : मुसीबत में मदद
Meaning : जो लोग लोग एक-दूसरे की तरह से प्रार्थना करते हैं, वह उनके बिना किसी भी व्यक्ति के साथ रहना चाहिए।
Doha    : जो तुम दोनों से करे, एक-दो लोग। बिना उसे दे दो कभी, केवल अपने संग ॥
----------------------------------------

Test 3:
Theme   : क्रोध
Context : गुस्से में झगड़ा
Meaning : यह दोहा नायिका के गुणों की शोभा का वर्णन करता है, जहाँ व्यक्ति ने मोहित हुए क्रीड़ा और गुणों के बीच से अत्यंत सुखों की तरह झगड़ा होता है।
Doha    : ज्यौं नसत कूप तुव सुरति, झट ल्याइ तिय। अलि जुगुक-बस

In [8]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import sentencepiece as spm
import math
import os
from huggingface_hub import hf_hub_download

# ==========================================
# 1. CONFIGURATION (कॉन्फ़िगरेशन)
# ==========================================
class Config:
    D_MODEL              = 256
    N_HEADS              = 8
    N_ENC_LAYERS         = 4
    N_MEANING_DEC_LAYERS = 4
    N_DOHA_DEC_LAYERS    = 4
    D_FF                 = 1024
    DROPOUT              = 0.15
    MAX_SEQ_LEN          = 256
    MAX_MEANING_LEN      = 60
    MAX_DOHA_LEN         = 48
    PAD_ID               = 0
    BOS_ID               = 2
    EOS_ID               = 3
    # Generation Params
    GEN_TEMPERATURE      = 0.8
    GEN_TOP_K            = 50
    GEN_TOP_P            = 0.92
    GEN_REP_PENALTY      = 1.3
    GEN_DOHA_REP_PEN     = 1.5

cfg = Config()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HF_REPO_ID = "nikpatidar333/doha-generation-model_v2" # Stage 2 v3 Repo

# ==========================================
# 2. MODEL ARCHITECTURE (मॉडल संरचना)
# ==========================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=512, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

class DohaDecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model); self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model); self.norm4 = nn.LayerNorm(d_model)
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.cross_attn_enc = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.cross_attn_meaning = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(d_ff, d_model), nn.Dropout(dropout))
        self.gate = nn.Parameter(torch.tensor(0.5))
    def forward(self, x, encoder_memory, meaning_memory, tgt_mask, enc_key_padding_mask, meaning_key_padding_mask):
        res = x; x = self.norm1(x)
        attn_out, _ = self.self_attn(x, x, x, attn_mask=tgt_mask)
        x = res + attn_out
        res = x; x_norm = self.norm2(x)
        enc_out, _ = self.cross_attn_enc(x_norm, encoder_memory, encoder_memory, key_padding_mask=enc_key_padding_mask)
        meaning_out, _ = self.cross_attn_meaning(x_norm, meaning_memory, meaning_memory, key_padding_mask=meaning_key_padding_mask)
        g = torch.sigmoid(self.gate)
        x = res + g * enc_out + (1.0 - g) * meaning_out
        res = x; x = self.norm3(x)
        x = res + self.ffn(x)
        return self.norm4(x)

class UnifiedDohaModel(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_enc_layers, n_meaning_dec_layers, n_doha_dec_layers, d_ff, dropout, max_len, pad_id):
        super().__init__()
        self.pad_id = pad_id; self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos_enc = PositionalEncoding(d_model, max_len, dropout)
        enc_l = nn.TransformerEncoderLayer(d_model, n_heads, d_ff, dropout, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(enc_l, num_layers=n_enc_layers)
        m_dec_l = nn.TransformerDecoderLayer(d_model, n_heads, d_ff, dropout, batch_first=True, norm_first=True)
        self.meaning_decoder = nn.TransformerDecoder(m_dec_l, num_layers=n_meaning_dec_layers)
        self.doha_dec_layers = nn.ModuleList([DohaDecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_doha_dec_layers)])
        self.doha_dec_norm = nn.LayerNorm(d_model)
        self.meaning_proj = nn.Linear(d_model, vocab_size, bias=False)
        self.doha_proj = nn.Linear(d_model, vocab_size, bias=False)
        self.meaning_proj.weight = self.embedding.weight
        self.doha_proj.weight = self.embedding.weight

    def encode(self, src, src_mask):
        x = self.pos_enc(self.embedding(src) * math.sqrt(self.d_model))
        return self.encoder(x, src_key_padding_mask=~src_mask)

    def decode_meaning(self, tgt, memory, src_mask):
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1)).to(tgt.device)
        x = self.pos_enc(self.embedding(tgt) * math.sqrt(self.d_model))
        return self.meaning_decoder(x, memory, tgt_mask=tgt_mask, memory_key_padding_mask=~src_mask)

    def decode_doha(self, tgt, enc_mem, mean_mem, enc_mask, mean_mask):
        tgt_mask = nn.Transformer.generate_square_subsequent_mask(tgt.size(1)).to(tgt.device)
        x = self.pos_enc(self.embedding(tgt) * math.sqrt(self.d_model))
        for layer in self.doha_dec_layers:
            x = layer(x, enc_mem, mean_mem, tgt_mask, ~enc_mask, ~mean_mask)
        return self.doha_dec_norm(x)

# ==========================================
# 3. DOWNLOAD & LOAD (डाउनलोड और लोड)
# ==========================================
print("--- Downloading Stage 2 v3 Model ---")
MODEL_PATH = hf_hub_download(repo_id=HF_REPO_ID, filename="best_model.pt")
TOKENIZER_PATH = hf_hub_download(repo_id=HF_REPO_ID, filename="tokenizer.model")

sp = spm.SentencePieceProcessor()
sp.load(TOKENIZER_PATH)
VOCAB_SIZE = sp.get_piece_size()
DANDAA_ID  = sp.piece_to_id('॥')
STOP_TOKENS = {cfg.EOS_ID, DANDAA_ID}

model = UnifiedDohaModel(VOCAB_SIZE, cfg.D_MODEL, cfg.N_HEADS, cfg.N_ENC_LAYERS, 
                         cfg.N_MEANING_DEC_LAYERS, cfg.N_DOHA_DEC_LAYERS, cfg.D_FF, 
                         cfg.DROPOUT, cfg.MAX_SEQ_LEN, cfg.PAD_ID).to(DEVICE)

ckpt = torch.load(MODEL_PATH, map_location=DEVICE)
# DataParallel handle
state_dict = {k.replace('module.', ''): v for k, v in ckpt['model_state'].items()}
model.load_state_dict(state_dict)
model.eval()
print(f"--- Model Loaded (Vocab: {VOCAB_SIZE}) ---")

# ==========================================
# 4. SAMPLING & GENERATION LOGIC
# ==========================================
def top_k_top_p_sample(logits, temperature=0.8, top_k=50, top_p=0.92, past_ids=None, rep_penalty=1.3):
    logits = logits.squeeze(0).float() / temperature
    if past_ids and rep_penalty > 1.0:
        for tid in set(past_ids[-20:]):
            logits[tid] /= rep_penalty if logits[tid] > 0 else (1/rep_penalty)
    if top_k > 0:
        v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
        logits[logits < v[-1]] = float('-inf')
    if top_p < 1.0:
        p_sort, p_idx = torch.sort(F.softmax(logits, dim=-1), descending=True)
        cumsum = torch.cumsum(p_sort, dim=-1)
        p_sort[(cumsum - p_sort) > top_p] = 0.0
        next_token = torch.multinomial(p_sort, 1)
        return p_idx.gather(-1, next_token)
    return torch.multinomial(F.softmax(logits, dim=-1), 1)

@torch.no_grad()
def generate(model, sp, theme, context):
    # Phase 1: Meaning
    enc_text = f"<theme> {theme} </theme> <context> {context} </context>"
    enc_ids = torch.tensor([sp.encode(enc_text)], dtype=torch.long).to(DEVICE)
    enc_mask = (enc_ids != cfg.PAD_ID)
    enc_mem = model.encode(enc_ids, enc_mask)
    
    mean_ids = [cfg.BOS_ID]
    for _ in range(cfg.MAX_MEANING_LEN):
        m_in = torch.tensor([mean_ids], dtype=torch.long).to(DEVICE)
        m_out = model.decode_meaning(m_in, enc_mem, enc_mask)
        next_id = top_k_top_p_sample(model.meaning_proj(m_out[:, -1, :]), 
                                     past_ids=mean_ids, rep_penalty=cfg.GEN_REP_PENALTY).item()
        if next_id == cfg.EOS_ID: break
        mean_ids.append(next_id)
    gen_meaning = sp.decode(mean_ids[1:])
    
    # Phase 2: Doha
    enc_text_full = f"{enc_text} <meaning> {gen_meaning} </meaning>"
    enc_ids_full = torch.tensor([sp.encode(enc_text_full)], dtype=torch.long).to(DEVICE)
    enc_mask_full = (enc_ids_full != cfg.PAD_ID)
    enc_mem_full = model.encode(enc_ids_full, enc_mask_full)
    
    m_mem = model.decode_meaning(torch.tensor([mean_ids], dtype=torch.long).to(DEVICE), 
                                  enc_mem_full, enc_mask_full)
    m_mask = (torch.tensor([mean_ids]) != cfg.PAD_ID).to(DEVICE)
    
    doha_ids = [cfg.BOS_ID]
    for _ in range(cfg.MAX_DOHA_LEN):
        d_in = torch.tensor([doha_ids], dtype=torch.long).to(DEVICE)
        d_out = model.decode_doha(d_in, enc_mem_full, m_mem, enc_mask_full, m_mask)
        next_id = top_k_top_p_sample(model.doha_proj(d_out[:, -1, :]), 
                                     past_ids=doha_ids, rep_penalty=cfg.GEN_DOHA_REP_PEN).item()
        doha_ids.append(next_id)
        if next_id in STOP_TOKENS: break
        
    return gen_meaning, sp.decode(doha_ids[1:])
# ── Matra counting ──────────────────────────────────────────
def count_matras(text):
    """Count matras in a Hindi text string."""
    # Long vowels — 2 matras
    long_vowels = set('आईऊएऐओऔाीूेैोौ')
    # Short vowels — 1 matra
    short_vowels = set('अइउऋिुृ')
    # Anusvara, visarga — 1 matra
    other_matras = set('ंःँ')

    count = 0
    for ch in text:
        if ch in long_vowels:
            count += 2
        elif ch in short_vowels or ch in other_matras:
            count += 1
        elif '\u0900' <= ch <= '\u097F':
            # Any other Devanagari char (consonants etc) = 1 matra
            count += 1
    return count


def matra_score(doha_text):
    """
    Returns (penalty, line1_matras, line2_matras).
    penalty=0 means perfect 24 matras per line.
    Lower penalty = better.
    """
    # Clean and split into lines
    text = doha_text.replace('॥', '').strip()
    # Try newline split first, then comma
    if '\n' in text:
        lines = [l.strip() for l in text.split('\n') if l.strip()]
    elif ',' in text:
        lines = [l.strip() for l in text.split(',') if l.strip()]
    else:
        lines = [text]

    if len(lines) < 2:
        return 999, 0, 0

    l1 = count_matras(lines[0])
    l2 = count_matras(lines[1])
    penalty = abs(l1 - 24) + abs(l2 - 24)
    return penalty, l1, l2


def generate_doha_best_of_n(model, sp, theme, context, n=5,
                              temperature=None, top_k=None, top_p=None):
    """
    Generate n dohas and return the one with best matra count.
    Target: 24 matras per line.
    """
    candidates = []
    for i in range(n):
        meaning, doha = generate_doha(
            model, sp, theme, context,
            temperature=temperature, top_k=top_k, top_p=top_p,
            verbose=False
        )
        penalty, l1, l2 = matra_score(doha)
        candidates.append({
            'doha'     : doha,
            'meaning'  : meaning,
            'penalty'  : penalty,
            'l1_matras': l1,
            'l2_matras': l2,
        })

    # Sort by matra penalty — lowest is best
    candidates.sort(key=lambda x: x['penalty'])
    best = candidates[0]

    print(f"थीम    : {theme}")
    print(f"संदर्भ : {context}")
    print(f"अर्थ   : {best['meaning']}")
    print(f"दोहा   : {best['doha']}")
    print(f"मात्रा : पंक्ति1={best['l1_matras']}/24  पंक्ति2={best['l2_matras']}/24  "
          f"दंड={best['penalty']}")
    print(f"सभी {n} प्रयास:")
    for j, c in enumerate(candidates):
        marker = '★' if j == 0 else ' '
        print(f"  {marker} [{j+1}] penalty={c['penalty']} "
              f"({c['l1_matras']}/{c['l2_matras']}) : {c['doha'][:60]}")
    print("-" * 60)

    return best


print("Matra scoring + best-of-N generation ready ✅")
# ==========================================
# 5. RUN BATCH TESTS (बैच टेस्ट)
# ==========================================
ntc = [
    # --- भक्ति और आध्यात्म (Devotion & Spirituality) ---
    {'theme': 'भक्ति', 'context': 'ईश्वर की आराधना'},
    {'theme': 'ईश्वर', 'context': 'कण-कण में भगवान'},
    {'theme': 'गुरु', 'context': 'गुरु की महिमा'},
    {'theme': 'मोक्ष', 'context': 'जीवन-मरण से मुक्ति'},
    {'theme': 'आनंद', 'context': 'प्रभु मिलन का सुख'},
    {'theme': 'कृष्ण', 'context': 'कान्हा की रासलीला'},
    {'theme': 'राम', 'context': 'मर्यादा पुरुषोत्तम राम'},
    {'theme': 'शिव', 'context': 'भोलेनाथ की कृपा'},
    {'theme': 'साधना', 'context': 'कठिन भक्ति मार्ग'},
    {'theme': 'रहस्य', 'context': 'परमात्मा का भेद'},

    # --- दर्शन और वैराग्य (Philosophy & Detachment) ---
    {'theme': 'ज्ञान', 'context': 'आत्मज्ञान की महिमा'},
    {'theme': 'वैराग्य', 'context': 'संसार से विरक्ति'},
    {'theme': 'माया', 'context': 'संसार का मोह'},
    {'theme': 'मृत्यु', 'context': 'जीवन का अंत'},
    {'theme': 'दर्शन', 'context': 'जीवन का सत्य'},
    {'theme': 'नश्वरता', 'context': 'मिट्टी का शरीर'},
    {'theme': 'मोह', 'context': 'मायाजाल में फँसना'},
    {'theme': 'त्याग', 'context': 'सुखों का बलिदान'},
    {'theme': 'संसार', 'context': 'दुनिया एक मेला'},
    {'theme': 'समय', 'context': 'वक्त की चाल'},

    # --- नीति और कर्म (Ethics & Karma) ---
    {'theme': 'नीति', 'context': 'जीवन के नियम'},
    {'theme': 'कर्म', 'context': 'कर्मों का फल'},
    {'theme': 'धर्म', 'context': 'सच्चा धर्म पालन'},
    {'theme': 'सत्य', 'context': 'सच्चाई की जीत'},
    {'theme': 'असत्य', 'context': 'झूठ का पर्दाफाश'},
    {'theme': 'पाप', 'context': 'बुरे कर्म का फल'},
    {'theme': 'पुण्य', 'context': 'अच्छे कर्मों का संचय'},
    {'theme': 'कर्मफल', 'context': 'किए का परिणाम'},
    {'theme': 'प्रारब्ध', 'context': 'पिछले जन्मों का फल'},
    {'theme': 'उपदेश', 'context': 'संतों की सीख'},

    # --- शृंगार और प्रेम (Beauty & Love) ---
    {'theme': 'शृंगार', 'context': 'नायिका का रूप'},
    {'theme': 'रूप', 'context': 'अद्भुत रूप लावण्य'},
    {'theme': 'सौंदर्य', 'context': 'प्राकृतिक सुंदरता'},
    {'theme': 'प्रेम', 'context': 'सच्चा निस्वार्थ प्रेम'},
    {'theme': 'विरह', 'context': 'प्रियतम से जुदाई'},
    {'theme': 'मिलन', 'context': 'प्रिय से भेंट'},
    {'theme': 'यौवन', 'context': 'जवानी का जोश'},
    {'theme': 'नारी', 'context': 'स्त्री का सौंदर्य'},
    {'theme': 'आकर्षण', 'context': 'रूप का जादू'},
    {'theme': 'छवि', 'context': 'मनमोहक सूरत'},

    # --- मानवीय गुण (Human Virtues) ---
    {'theme': 'संतोष', 'context': 'परम सुख संतोष'},
    {'theme': 'धैर्य', 'context': 'मुसीबत में धीरज'},
    {'theme': 'क्षमा', 'context': 'गलती माफ करना'},
    {'theme': 'दया', 'context': 'जीवों पर कृपा'},
    {'theme': 'परोपकार', 'context': 'निस्वार्थ सेवा'},
    {'theme': 'दान', 'context': 'निस्वार्थ गुप्त दान'},
    {'theme': 'साहस', 'context': 'निडर होकर लड़ना'},
    {'theme': 'संयम', 'context': 'इंद्रियों पर काबू'},
    {'theme': 'विनम्रता', 'context': 'बड़ों का आदर'},
    {'theme': 'परमार्थ', 'context': 'दूसरों की भलाई'},

    # --- मानवीय अवगुण (Human Vices) ---
    {'theme': 'अहंकार', 'context': 'झूठा घमंड करना'},
    {'theme': 'लोभ', 'context': 'लालच बुरी बला'},
    {'theme': 'क्रोध', 'context': 'गुस्से का नुकसान'},
    {'theme': 'अज्ञान', 'context': 'मूर्खता का परिणाम'},
    {'theme': 'स्वार्थ', 'context': 'अपना मतलब देखना'},
    {'theme': 'क्रूरता', 'context': 'निर्दयी व्यवहार'},
    {'theme': 'कायरता', 'context': 'डर कर भागना'},
    {'theme': 'धोखा', 'context': 'छल-कपट करना'},
    {'theme': 'अभिमान', 'context': 'झूठी शान'},
    {'theme': 'ईर्ष्या', 'context': 'दूसरों से जलना'},

    # --- जीवन की अवस्थाएं और स्थितियां (Life Stages & Situations) ---
    {'theme': 'जीवन', 'context': 'जीवन का संघर्ष'},
    {'theme': 'बचपन', 'context': 'मासूमियत का खेल'},
    {'theme': 'बुढ़ापा', 'context': 'उम्र का ढलना'},
    {'theme': 'गरीबी', 'context': 'निर्धनता का दुख'},
    {'theme': 'अमीरी', 'context': 'धन का अहंकार'},
    {'theme': 'सुख', 'context': 'जीवन में खुशहाली'},
    {'theme': 'दुख', 'context': 'पीड़ा और कष्ट'},
    {'theme': 'भाग्य', 'context': 'किस्मत का खेल'},
    {'theme': 'सफलता', 'context': 'मंजिल का मिलना'},
    {'theme': 'विफलता', 'context': 'हार का सामना'},

    # --- समाज और व्यवहार (Society & Behavior) ---
    {'theme': 'समाज', 'context': 'सामाजिक नियम'},
    {'theme': 'सत्संग', 'context': 'अच्छी संगति का असर'},
    {'theme': 'कुसंग', 'context': 'बुरी संगति का फल'},
    {'theme': 'मित्रता', 'context': 'सच्चे दोस्त की पहचान'},
    {'theme': 'शत्रुता', 'context': 'आपसी बैर भाव'},
    {'theme': 'राजनीति', 'context': 'सत्ता का खेल'},
    {'theme': 'न्याय', 'context': 'सबके साथ इंसाफ'},
    {'theme': 'अन्याय', 'context': 'कमजोर पर जुल्म'},
    {'theme': 'शिक्षा', 'context': 'विद्या का महत्व'},
    {'theme': 'स्वाभिमान', 'context': 'आत्मसम्मान की रक्षा'},

    # --- मन, वाणी और विचार (Mind, Speech & Thoughts) ---
    {'theme': 'मन', 'context': 'मन की चंचलता'},
    {'theme': 'वाणी', 'context': 'मीठे बोल'},
    {'theme': 'मौन', 'context': 'चुप रहने का सुख'},
    {'theme': 'आशा', 'context': 'उम्मीद की किरण'},
    {'theme': 'निराशा', 'context': 'हताशा और दुख'},
    {'theme': 'चिंता', 'context': 'भविष्य की फिक्र'},
    {'theme': 'चिंतन', 'context': 'गहरा विचार करना'},
    {'theme': 'संकल्प', 'context': 'पक्का इरादा'},
    {'theme': 'विकल्प', 'context': 'रास्ते की दुविधा'},
    {'theme': 'विश्वास', 'context': 'अटूट भरोसा'},

    # --- प्रकृति और अन्य (Nature & Miscellaneous) ---
    {'theme': 'प्रकृति', 'context': 'वसंत की बहार'},
    {'theme': 'ऋतु', 'context': 'सावन की फुहार'},
    {'theme': 'वीर', 'context': 'युद्ध में साहस'},
    {'theme': 'करुण', 'context': 'दुखियों पर दया'},
    {'theme': 'शांत', 'context': 'मन की शांति'},
    {'theme': 'पुरुष', 'context': 'पुरुष का पौरुष'},
    {'theme': 'घर', 'context': 'सुखी परिवार'},
    {'theme': 'उत्सव', 'context': 'त्योहार की खुशी'},
    {'theme': 'विवाद', 'context': 'आपसी कलह'},
    {'theme': 'संवाद', 'context': 'प्रेम से बातचीत'}
]

print("\n" + "="*60)
print(" RUNNING STAGE 2 v3 BATCH TESTS ")
print("="*60)

for i, test in enumerate(ntc):
    meaning, doha = generate(model, sp, test['theme'], test['context'])
    print(f"\nTest {i+1}:")
    print(f"Theme   : {test['theme']}")
    print(f"Context : {test['context']}")
    print(f"Meaning : {meaning}")
    print(f"Doha    : {doha}")
    print("-" * 40)

print("\nBatch testing complete. ✅")

--- Downloading Stage 2 v3 Model ---


/tmp/ipykernel_55/3326802093.py:84: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.encoder = nn.TransformerEncoder(enc_l, num_layers=n_enc_layers)


--- Model Loaded (Vocab: 8000) ---
Matra scoring + best-of-N generation ready ✅

 RUNNING STAGE 2 v3 BATCH TESTS 

Test 1:
Theme   : भक्ति
Context : ईश्वर की आराधना
Meaning : प्रभु (ईश्वर) से ईश्वर का पालन करता है, पर धनुष का मानकर स्वयं ही मन को हर लेती है।
Doha    : ईश्वर का पालन से, मन है हर मान। धनुष समान स्वयं को, एक तन पर मन भगवान ॥
----------------------------------------

Test 2:
Theme   : ईश्वर
Context : कण-कण में भगवान
Meaning : यह दोहा एक कण-कण में है, और इसके इस प्रकार का वर्णन करता है कि संसार संसार में कोई स्थान नहीं देता।
Doha    : एक कण-कण में, कोऊ न पाताल। यह संसार में रहै, रूप न कोई स्थान ॥
----------------------------------------

Test 3:
Theme   : गुरु
Context : गुरु की महिमा
Meaning : गुरु के गुरु का पालन कर रहा है और संतों के संग घूम रहे हैं, परंतुंतु उसे कभी गुरु की महिमा का रूप नहीं मिलता।
Doha    : गुरु के गुरु पालन चलत, संगति संग्राम । परग न मिलै जो नहीं कभी, सतगुरु की रूप ॥
----------------------------------------

Test 4:
Theme   : मोक्ष
Context : जीवन-मरण स